# GeoAgent — Autonomous Reservoir Property Translation
**Author: Abdulrahman Al-Fakih**

Bidirectional Pix2Pix GAN wrapped in a ReAct agentic loop.

**Datasets (local):**
- `24Paired_Porosity_Facies_split` — Porosity ↔ Facies
- `Paired_PERM_Facies_split0` — Facies → Permeability

| Dataset | Directions | Est. time (GPU) |
|---------|-----------|------------------|
| Porosity ↔ Facies | AB + BA | ~7 hrs |
| Facies → Permeability | AB | ~3.5 hrs |

## 0. Verify GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU:", name)
    print("VRAM: {:.1f} GB".format(vram))
else:
    print("WARNING: No GPU detected.")


## 1. Install Dependencies

*(Kernel restarts automatically after pip — normal. Re-run from Cell 0.)*

In [ ]:
!pip install torch torchvision pillow scikit-image matplotlib tqdm
print("Done.")


## 2. Write Code Files

*Run once — writes the 5 helper `.py` files next to this notebook.*

In [ ]:
%%writefile geoagent_dataset.py
"""Dataset loader. Author: Abdulrahman Al-Fakih"""
import os, re, torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

def get_transform(image_size=256, augment=True):
    ops = []
    if augment:
        ops += [transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.2)]
    ops += [transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])]
    return transforms.Compose(ops)

class PairedGeoDataset(Dataset):
    def __init__(self, root_dir, direction="AB", image_size=256, augment=True):
        super().__init__()
        self.root_dir  = root_dir
        self.direction = direction
        self.image_size = image_size
        self.files = sorted([os.path.join(root_dir, f)
                             for f in os.listdir(root_dir)
                             if f.lower().endswith(".png")])
        assert len(self.files) > 0, "No PNG files in " + root_dir
        self.transform = get_transform(image_size, augment)
        print("[Dataset]", len(self.files), "images | direction=" + direction)

    def __len__(self): return len(self.files)

    def _apply_transform(self, imgA, imgB):
        seed = torch.randint(0, 2**32, (1,)).item()
        torch.manual_seed(seed); tA = self.transform(imgA)
        torch.manual_seed(seed); tB = self.transform(imgB)
        return tA, tB

    def __getitem__(self, idx):
        path = self.files[idx]
        img  = Image.open(path).convert("RGB")
        w, h = img.size
        mid  = w // 2
        A = img.crop((0,   0, mid, h))
        B = img.crop((mid, 0, w,   h))
        tA, tB = self._apply_transform(A, B)
        m = re.search(r"cube_i(\d+)_j(\d+)_k(\d+)", os.path.basename(path))
        coords = (int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else (-1,-1,-1)
        if self.direction == "AB":
            return {"input": tA, "target": tB, "path": path, "coords": coords}
        elif self.direction == "BA":
            return {"input": tB, "target": tA, "path": path, "coords": coords}
        return {"A": tA, "B": tB, "path": path, "coords": coords}

def build_dataloaders(data_root, direction="AB", image_size=256,
                      batch_size=4, num_workers=2):
    loaders = {}
    for split in ("train", "val", "test"):
        path = os.path.join(data_root, split)
        if not os.path.exists(path):
            raise FileNotFoundError("Split not found: " + path)
        ds = PairedGeoDataset(path, direction=direction,
                              image_size=image_size, augment=(split=="train"))
        loaders[split] = DataLoader(ds, batch_size=batch_size,
                                    shuffle=(split=="train"),
                                    num_workers=num_workers,
                                    pin_memory=torch.cuda.is_available(),
                                    drop_last=(split=="train"))
    return loaders


In [ ]:
# Cell: Add 3D triplet loader to geoagent_dataset.py
append_code = '''

# ── 3D Triplet Dataset ────────────────────────────────────────────────────────
from glob import glob as _glob

class PairedDataset3D(torch.utils.data.Dataset):
    """9-channel input: concat(slice i-1, i, i+1) → 3-channel target (slice i)."""

    def __init__(self, root, direction, split, image_size=256):
        self.direction = direction
        tfm = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])
        self.tfm = tfm

        a_dir = os.path.join(root, split, "A")
        b_dir = os.path.join(root, split, "B")
        all_a = sorted(_glob(os.path.join(a_dir, "*.png")))
        all_b = sorted(_glob(os.path.join(b_dir, "*.png")))
        # Use only middle slices (need i-1 and i+1)
        self.a_files = all_a[1:-1]
        self.b_files = all_b[1:-1]
        self.all_a   = all_a   # needed to access neighbors
        self.all_b   = all_b

    def __len__(self):
        return len(self.a_files)

    def __getitem__(self, idx):
        # idx in self.a_files corresponds to idx+1 in self.all_a
        real_idx = idx + 1
        def load(path):
            return self.tfm(Image.open(path).convert("RGB"))

        prev = load(self.all_a[real_idx - 1])
        curr = load(self.all_a[real_idx])
        nxt  = load(self.all_a[real_idx + 1])
        inp_9ch = torch.cat([prev, curr, nxt], dim=0)   # (9, H, W)

        if self.direction == "AB":
            inp = inp_9ch
            tgt = load(self.all_b[real_idx])
        else:  # BA
            # For BA: B is input (3ch → pad to 9ch), A is target
            b_prev = load(self.all_b[real_idx - 1])
            b_curr = load(self.all_b[real_idx])
            b_nxt  = load(self.all_b[real_idx + 1])
            inp = torch.cat([b_prev, b_curr, b_nxt], dim=0)
            tgt = load(self.all_a[real_idx])

        return {"input": inp, "target": tgt}


def build_3d_dataloaders(data_root, direction, image_size=256,
                          batch_size=4, num_workers=0):
    loaders = {}
    for split in ("train", "val", "test"):
        ds = PairedDataset3D(data_root, direction, split, image_size)
        loaders[split] = torch.utils.data.DataLoader(
            ds, batch_size=batch_size,
            shuffle=(split == "train"), num_workers=num_workers,
            drop_last=(split == "train"),
        )
        print(f"[3D Dataset] {len(ds)} triplets | direction={direction} | split={split}")
    return loaders
'''

with open(r"C:\Users\Abdulrahman.fakih\geoagent_dataset.py", "a", encoding="utf-8") as f:
    f.write(append_code)
print("geoagent_dataset.py updated with 3D loader")

In [ ]:
%%writefile geoagent_model.py
"""Pix2Pix model — all inplace=False. Author: Abdulrahman Al-Fakih"""
import torch
import torch.nn as nn

class ConvNormReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=4, stride=2, padding=1,
                 norm="batch", activation="leaky", transpose=False):
        super().__init__()
        layers = []
        bias = (norm == "none")
        if transpose:
            layers.append(nn.ConvTranspose2d(in_ch, out_ch, kernel, stride, padding, bias=bias))
        else:
            layers.append(nn.Conv2d(in_ch, out_ch, kernel, stride, padding, bias=bias))
        if norm == "batch":      layers.append(nn.BatchNorm2d(out_ch))
        elif norm == "instance": layers.append(nn.InstanceNorm2d(out_ch))
        if activation == "leaky": layers.append(nn.LeakyReLU(0.2, inplace=False))
        elif activation == "relu": layers.append(nn.ReLU(inplace=False))
        elif activation == "tanh": layers.append(nn.Tanh())
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class UNetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64, norm="batch"):
        super().__init__()
        # Encoder
        self.e1 = nn.Conv2d(in_ch, ngf, 4, 2, 1, bias=False)
        self.e2 = ConvNormReLU(ngf,   ngf*2, norm=norm)
        self.e3 = ConvNormReLU(ngf*2, ngf*4, norm=norm)
        self.e4 = ConvNormReLU(ngf*4, ngf*8, norm=norm)
        self.e5 = ConvNormReLU(ngf*8, ngf*8, norm=norm)
        self.e6 = ConvNormReLU(ngf*8, ngf*8, norm=norm)
        self.e7 = ConvNormReLU(ngf*8, ngf*8, norm=norm)
        self.e8 = nn.Sequential(
            nn.LeakyReLU(0.2, inplace=False),
            nn.Conv2d(ngf*8, ngf*8, 4, 2, 1, bias=False))
        # Decoder
        self.d8 = self._dec(ngf*8,  ngf*8,  drop=True,  norm=norm)
        self.d7 = self._dec(ngf*16, ngf*8,  drop=True,  norm=norm)
        self.d6 = self._dec(ngf*16, ngf*8,  drop=True,  norm=norm)
        self.d5 = self._dec(ngf*16, ngf*8,  drop=False, norm=norm)
        self.d4 = self._dec(ngf*16, ngf*4,  drop=False, norm=norm)
        self.d3 = self._dec(ngf*8,  ngf*2,  drop=False, norm=norm)
        self.d2 = self._dec(ngf*4,  ngf,    drop=False, norm=norm)
        self.d1 = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.ConvTranspose2d(ngf*2, out_ch, 4, 2, 1, bias=False),
            nn.Tanh())
    @staticmethod
    def _dec(in_ch, out_ch, drop=False, norm="batch"):
        layers = [nn.ReLU(inplace=False),
                  nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if norm == "batch":      layers.append(nn.BatchNorm2d(out_ch))
        elif norm == "instance": layers.append(nn.InstanceNorm2d(out_ch))
        if drop: layers.append(nn.Dropout(0.5))
        return nn.Sequential(*layers)
    def forward(self, x):
        e1=self.e1(x); e2=self.e2(e1); e3=self.e3(e2); e4=self.e4(e3)
        e5=self.e5(e4); e6=self.e6(e5); e7=self.e7(e6); e8=self.e8(e7)
        d8=self.d8(e8)
        d7=self.d7(torch.cat([d8,e7],1))
        d6=self.d6(torch.cat([d7,e6],1))
        d5=self.d5(torch.cat([d6,e5],1))
        d4=self.d4(torch.cat([d5,e4],1))
        d3=self.d3(torch.cat([d4,e3],1))
        d2=self.d2(torch.cat([d3,e2],1))
        return self.d1(torch.cat([d2,e1],1))

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_ch=6, ndf=64, n_layers=3, norm="batch"):
        super().__init__()
        NL = nn.BatchNorm2d if norm == "batch" else nn.InstanceNorm2d
        seq = [nn.Conv2d(in_ch,ndf,4,2,1,bias=False), nn.LeakyReLU(0.2,inplace=False)]
        ch = ndf
        for n in range(1, n_layers):
            ch_prev, ch = ch, min(ch*2, 512)
            stride = 1 if n == n_layers-1 else 2
            seq += [nn.Conv2d(ch_prev,ch,4,stride,1,bias=False), NL(ch),
                    nn.LeakyReLU(0.2,inplace=False)]
        ch2 = min(ch*2, 512)
        seq += [nn.Conv2d(ch,ch2,4,1,1,bias=False), NL(ch2),
                nn.LeakyReLU(0.2,inplace=False),
                nn.Conv2d(ch2,1,4,1,1)]
        self.model = nn.Sequential(*seq)
    def forward(self, inp, tgt):
        return self.model(torch.cat([inp,tgt],1))

class Pix2PixModel:
    def __init__(self, device, in_ch=3, out_ch=3, ngf=64, ndf=64,
                 lambda_l1=100.0, norm="batch", lr=2e-4, beta1=0.5):
        self.device = device
        self.lambda_l1 = lambda_l1
        self.G = UNetGenerator(in_ch, out_ch, ngf, norm).to(device)
        self.D = PatchGANDiscriminator(in_ch+out_ch, ndf, norm=norm).to(device)
        self.cGAN = nn.BCEWithLogitsLoss()
        self.cL1  = nn.L1Loss()
        self.oG = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(beta1,0.999))
        self.oD = torch.optim.Adam(self.D.parameters(), lr=lr, betas=(beta1,0.999))
        self.sG = torch.optim.lr_scheduler.LambdaLR(
            self.oG, lambda e: max(0.0, 1.0-max(0,e-100)/100.0))
        self.sD = torch.optim.lr_scheduler.LambdaLR(
            self.oD, lambda e: max(0.0, 1.0-max(0,e-100)/100.0))
        self._init_weights()
    def _init_weights(self):
        for m in list(self.G.modules())+list(self.D.modules()):
            if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d)):
                nn.init.normal_(m.weight,0.0,0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m,(nn.BatchNorm2d,nn.InstanceNorm2d)):
                if m.weight is not None: nn.init.normal_(m.weight,1.0,0.02)
                if m.bias  is not None: nn.init.zeros_(m.bias)
    def _rg(self, net, flag):
        for p in net.parameters(): p.requires_grad = flag
    def train_step(self, rA, rB):
        rA, rB = rA.to(self.device), rB.to(self.device)
        fB = self.G(rA)
        self._rg(self.D, True); self.oD.zero_grad()
        pf = self.D(rA, fB.detach())
        pr = self.D(rA, rB)
        lD = 0.5*(self.cGAN(pf,torch.zeros_like(pf))+self.cGAN(pr,torch.ones_like(pr)))
        lD.backward(); self.oD.step()
        self._rg(self.D, False); self.oG.zero_grad()
        pg  = self.D(rA, fB)
        lGg = self.cGAN(pg, torch.ones_like(pg))
        lGl = self.cL1(fB, rB) * self.lambda_l1
        lG  = lGg + lGl; lG.backward(); self.oG.step()
        return {"loss_G":lG.item(),"loss_G_gan":lGg.item(),
                "loss_G_l1":lGl.item(),"loss_D":lD.item()}
    @torch.no_grad()
    def predict(self, x):
        self.G.eval()
        return self.G(x.to(self.device))
    def step_schedulers(self): self.sG.step(); self.sD.step()
    def save(self, path):
        torch.save({"G":self.G.state_dict(),"D":self.D.state_dict()}, path)
        print("Saved:", path)
    def load(self, path):
        c = torch.load(path, map_location=self.device)
        self.G.load_state_dict(c["G"]); self.D.load_state_dict(c["D"])
        print("Loaded:", path)

def count_params(model):
    return "  |  ".join(
        "{}: {:.2f}M params".format(n, sum(p.numel() for p in net.parameters())/1e6)
        for n, net in [("G", model.G), ("D", model.D)])


In [ ]:
%%writefile geoagent_train.py
"""Training script. Author: Abdulrahman Al-Fakih"""
import os, argparse, time, json
import torch
from torchvision.utils import save_image
from geoagent_dataset import build_dataloaders
from geoagent_model import Pix2PixModel, count_params

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data_root",   required=True)
    p.add_argument("--direction",   default="AB", choices=["AB","BA"])
    p.add_argument("--epochs",      type=int,   default=200)
    p.add_argument("--batch_size",  type=int,   default=4)
    p.add_argument("--image_size",  type=int,   default=256)
    p.add_argument("--ngf",         type=int,   default=64)
    p.add_argument("--ndf",         type=int,   default=64)
    p.add_argument("--lambda_l1",   type=float, default=100.0)
    p.add_argument("--lr",          type=float, default=2e-4)
    p.add_argument("--norm",        default="batch")
    p.add_argument("--save_dir",    required=True)
    p.add_argument("--save_freq",   type=int,   default=10)
    p.add_argument("--vis_freq",    type=int,   default=5)
    p.add_argument("--num_workers", type=int,   default=0)
    p.add_argument("--resume",      default=None)
    return p.parse_args()

def denorm(t): return (t.clamp(-1,1)+1)/2

def save_samples(model, loader, epoch, out_dir, n=4):
    os.makedirs(out_dir, exist_ok=True)
    batch = next(iter(loader))
    inp = batch["input"][:n]; tgt = batch["target"][:n]
    gen = model.predict(inp).cpu()
    grid = torch.cat([denorm(inp), denorm(gen), denorm(tgt)], dim=-1)
    save_image(grid, os.path.join(out_dir, "epoch_{:04d}.png".format(epoch)), nrow=1)

def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Direction:", args.direction, "| Device:", str(device), "| Epochs:", args.epochs)
    loaders = build_dataloaders(args.data_root, args.direction,
                                args.image_size, args.batch_size, args.num_workers)
    model = Pix2PixModel(device, ngf=args.ngf, ndf=args.ndf,
                         lambda_l1=args.lambda_l1, norm=args.norm, lr=args.lr)
    print(count_params(model))
    start_epoch = 1
    if args.resume and os.path.exists(args.resume):
        model.load(args.resume)
        try:
            start_epoch = int(os.path.basename(args.resume).split("ep")[1].split(".")[0]) + 1
        except Exception:
            start_epoch = 1
    os.makedirs(args.save_dir, exist_ok=True)
    vis_dir  = os.path.join(args.save_dir, "samples")
    log_path = os.path.join(args.save_dir, "training_log_" + args.direction + ".json")
    history  = []
    for epoch in range(start_epoch, args.epochs+1):
        model.G.train(); model.D.train()
        totals = {"loss_G":0,"loss_G_gan":0,"loss_G_l1":0,"loss_D":0}
        t0 = time.time()
        for batch in loaders["train"]:
            ls = model.train_step(batch["input"], batch["target"])
            for k in totals: totals[k] += ls[k]
        n = len(loaders["train"])
        avg = {k: round(v/n,4) for k,v in totals.items()}
        elapsed = round(time.time()-t0, 1)
        print("Epoch [{:4d}] G={:.4f} G_gan={:.4f} G_l1={:.4f} D={:.4f} ({:.1f}s)".format(
              epoch, avg["loss_G"], avg["loss_G_gan"], avg["loss_G_l1"], avg["loss_D"], elapsed))
        model.step_schedulers()
        history.append({"epoch":epoch,"elapsed_s":elapsed,**avg})
        if epoch % args.vis_freq == 0:
            save_samples(model, loaders["val"], epoch, vis_dir)
        if epoch % args.save_freq == 0 or epoch == args.epochs:
            name = "pix2pix_{}_ep{:04d}.pth".format(args.direction, epoch)
            model.save(os.path.join(args.save_dir, name))
        with open(log_path,"w") as f: json.dump(history, f, indent=2)
    print("Training done. Saved to:", args.save_dir)

if __name__ == "__main__":
    train(parse_args())


In [ ]:
%%writefile geoagent_agent.py
"""GeoAgent ReAct loop. Author: Abdulrahman Al-Fakih"""
import os, time
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from dataclasses import dataclass, field
from typing import Optional
from geoagent_model import Pix2PixModel

@dataclass
class AgentStep:
    step: int
    action: str
    observation: str
    ssim: Optional[float] = None
    accepted: Optional[bool] = None

@dataclass
class AgentResult:
    predicted_image: torch.Tensor
    direction: str
    confidence: float
    ssim_score: float
    n_iterations: int
    accepted: bool
    reasoning_trace: list = field(default_factory=list)
    elapsed_ms: float = 0.0
    def summary(self):
        labels = {"AB":"Porosity->Facies","BA":"Facies->Porosity","FP":"Facies->Permeability"}
        label  = labels.get(self.direction, self.direction)
        status = "ACCEPTED" if self.accepted else "BEST AVAILABLE"
        return ("GeoAgent Result" + chr(10) +
                "  Direction  : " + label + chr(10) +
                "  SSIM score : {:.4f}".format(self.ssim_score) + chr(10) +
                "  Confidence : {:.3f}".format(self.confidence) + chr(10) +
                "  Iterations : {:d}".format(self.n_iterations) + chr(10) +
                "  Status     : " + status + chr(10) +
                "  Time (ms)  : {:.1f}".format(self.elapsed_ms))

def load_image_tensor(path, image_size=256):
    tfm = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])
    return tfm(Image.open(path).convert("RGB")).unsqueeze(0)

def tensor_to_pil(t):
    arr = ((t.squeeze(0).permute(1,2,0).cpu().numpy()+1)/2*255).clip(0,255).astype(np.uint8)
    return Image.fromarray(arr)

def compute_ssim(pred, target):
    p = (pred.squeeze(0).cpu().numpy()+1)/2
    t = (target.squeeze(0).cpu().numpy()+1)/2
    C1, C2 = 0.01**2, 0.03**2
    mu_p, mu_t = p.mean(), t.mean()
    sp, st = p.std(), t.std()
    spt = ((p-mu_p)*(t-mu_t)).mean()
    return float(((2*mu_p*mu_t+C1)*(2*spt+C2)) /
                 ((mu_p**2+mu_t**2+C1)*(sp**2+st**2+C2)))

def compute_pixel_accuracy(pred, target, tol=0.1):
    return float(((pred.cpu()-target.cpu()).abs()<tol).float().mean())

TTA = [
    (lambda x: x,                        lambda x: x),
    (lambda x: torch.flip(x,[-1]),        lambda x: torch.flip(x,[-1])),
    (lambda x: torch.flip(x,[-2]),        lambda x: torch.flip(x,[-2])),
    (lambda x: torch.flip(x,[-1,-2]),     lambda x: torch.flip(x,[-1,-2])),
    (lambda x: torch.rot90(x,1,[-2,-1]),  lambda x: torch.rot90(x,-1,[-2,-1])),
]

def tta_predict(model, inp, n_aug=5):
    return torch.stack([inv(model.predict(fwd(inp))) for fwd,inv in TTA[:n_aug]]).mean(0)

class GeoAgent:
    def __init__(self, model_AB_path, model_BA_path,
                 device=None, ssim_threshold=0.70, max_iterations=3,
                 image_size=256, verbose=True):
        self.device        = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.ssim_threshold = ssim_threshold
        self.max_iterations = max_iterations
        self.image_size    = image_size
        self.verbose       = verbose
        self.model_AB = Pix2PixModel(device=self.device)
        self.model_AB.load(model_AB_path)
        self.model_BA = Pix2PixModel(device=self.device)
        self.model_BA.load(model_BA_path)
        if verbose:
            print("GeoAgent ready | device=" + str(self.device) +
                  " | threshold=" + str(ssim_threshold))
    def _log(self, msg):
        if self.verbose: print("  [Agent]", msg)
    def run(self, input_tensor, target_tensor=None, direction="AB"):
        t0    = time.time()
        trace = []
        step  = 0
        model = self.model_AB if direction == "AB" else self.model_BA
        step += 1
        obs = "Direction=" + direction
        self._log("Step {} [REASON] {}".format(step, obs))
        trace.append(AgentStep(step=step, action="set_direction", observation=obs))
        step += 1
        pred = model.predict(input_tensor)
        obs  = "Initial prediction generated"
        self._log("Step {} [ACT]    {}".format(step, obs))
        trace.append(AgentStep(step=step, action="translate", observation=obs))
        step += 1
        ssim     = compute_ssim(pred, target_tensor) if target_tensor is not None else 0.5
        accepted = ssim >= self.ssim_threshold
        obs = "SSIM={:.4f} thr={:.2f} accepted={}".format(ssim, self.ssim_threshold, accepted)
        self._log("Step {} [OBSERVE] {}".format(step, obs))
        trace.append(AgentStep(step=step, action="evaluate",
                               observation=obs, ssim=ssim, accepted=accepted))
        best_pred, best_ssim = pred, ssim
        n_iter = 1
        while not accepted and n_iter < self.max_iterations:
            n_iter += 1; step += 1
            obs = "Below threshold — TTA iter {}".format(n_iter)
            self._log("Step {} [REASON] {}".format(step, obs))
            trace.append(AgentStep(step=step, action="plan_tta", observation=obs))
            step += 1
            pred_tta = tta_predict(model, input_tensor, n_aug=3+n_iter)
            obs = "TTA done"
            self._log("Step {} [ACT]    {}".format(step, obs))
            trace.append(AgentStep(step=step, action="tta_translate", observation=obs))
            step += 1
            ssim_tta = compute_ssim(pred_tta, target_tensor) if target_tensor is not None else 0.5
            accepted = ssim_tta >= self.ssim_threshold
            if ssim_tta > best_ssim: best_pred, best_ssim = pred_tta, ssim_tta
            obs = "TTA SSIM={:.4f} best={:.4f} accepted={}".format(
                  ssim_tta, best_ssim, accepted)
            self._log("Step {} [OBSERVE] {}".format(step, obs))
            trace.append(AgentStep(step=step, action="evaluate",
                                   observation=obs, ssim=ssim_tta, accepted=accepted))
        confidence = float(min(best_ssim / self.ssim_threshold, 1.0))
        elapsed_ms = (time.time()-t0)*1000
        self._log("Done {:.0f}ms SSIM={:.4f} conf={:.3f}".format(
                  elapsed_ms, best_ssim, confidence))
        return AgentResult(predicted_image=best_pred, direction=direction,
                           confidence=confidence, ssim_score=best_ssim,
                           n_iterations=n_iter, accepted=accepted,
                           reasoning_trace=trace, elapsed_ms=elapsed_ms)
    def run_from_path(self, input_path, target_path=None, direction="AB"):
        inp = load_image_tensor(input_path, self.image_size)
        tgt = load_image_tensor(target_path, self.image_size) if target_path else None
        return self.run(inp, tgt, direction)


In [ ]:
%%writefile geoagent_evaluate.py
"""Evaluation. Author: Abdulrahman Al-Fakih"""
import os, csv, json, argparse
import torch, numpy as np
from torchvision.utils import save_image
from geoagent_dataset import build_dataloaders
from geoagent_agent import GeoAgent, compute_ssim, compute_pixel_accuracy

def compute_psnr(pred, target):
    p = (pred.cpu().clamp(-1,1)+1)/2
    t = (target.cpu().clamp(-1,1)+1)/2
    mse = ((p-t)**2).mean().item()
    return 100.0 if mse < 1e-10 else float(10*np.log10(1.0/mse))

def compute_mae(pred, target):
    p = (pred.cpu().clamp(-1,1)+1)/2
    t = (target.cpu().clamp(-1,1)+1)/2
    return float((p-t).abs().mean())

def evaluate(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("=" * 55)
    print("  GeoAgent Eval | dir=" + args.direction + " | split=" + args.split)
    print("=" * 55)
    os.makedirs(args.output_dir, exist_ok=True)
    loaders = build_dataloaders(args.data_root, args.direction,
                                args.image_size, 1, 0)
    loader  = loaders[args.split]
    agent   = GeoAgent(args.model_AB, args.model_BA, device,
                       args.ssim_threshold, args.max_iterations,
                       args.image_size, verbose=False)
    rows, preds, inps, tgts = [], [], [], []
    for idx, batch in enumerate(loader):
        inp, tgt = batch["input"], batch["target"]
        # FIXED (handles both 2D and 3D):
        raw = batch.get("coords", None)
        coords = tuple(int(c[0]) for c in raw) if raw is not None else (0, 0, 0)
        coord_str = "i{}_j{}_k{}".format(coords[0], coords[1], coords[2])
        result = agent.run(inp, tgt, direction=args.direction)
        pred   = result.predicted_image.cpu()
        rows.append({
            "idx":       idx,
            "filename":  os.path.basename(batch["path"][0]),
            "i": coords[0], "j": coords[1], "k": coords[2],
            "ssim":      round(compute_ssim(pred,tgt),4),
            "psnr_db":   round(compute_psnr(pred,tgt),2),
            "mae":       round(compute_mae(pred,tgt),4),
            "pixel_acc": round(compute_pixel_accuracy(pred,tgt),4),
            "n_iter":    result.n_iterations,
            "accepted":  result.accepted,
            "confidence":round(result.confidence,3),
            "elapsed_ms":round(result.elapsed_ms,1),
        })
        preds.append(pred); inps.append(inp); tgts.append(tgt)
        if (idx+1) % 25 == 0 or (idx+1) == len(loader):
            ms = round(np.mean([r["ssim"] for r in rows]),4)
            print("  [{}/{}]  mean SSIM={:.4f}".format(idx+1,len(loader),ms))
    mk = ["ssim","psnr_db","mae","pixel_acc","n_iter","elapsed_ms"]
    summary = {m:{"mean":round(float(np.mean([r[m] for r in rows])),4),
                  "std": round(float(np.std ([r[m] for r in rows])),4)} for m in mk}
    summary["acceptance_rate"] = round(sum(r["accepted"] for r in rows)/len(rows),4)
    summary["n_samples"] = len(rows)
    summary["direction"] = args.direction
    print("SSIM={:.4f}+/-{:.4f}  PSNR={:.2f}dB  Accept={:.1f}%".format(
          summary["ssim"]["mean"], summary["ssim"]["std"],
          summary["psnr_db"]["mean"], summary["acceptance_rate"]*100))
    tag = args.direction + "_" + args.split
    with open(os.path.join(args.output_dir,"metrics_"+tag+".csv"),"w",newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader(); w.writerows(rows)
    with open(os.path.join(args.output_dir,"summary_"+tag+".json"),"w") as f:
        json.dump(summary,f,indent=2)
    denorm = lambda t: (t.clamp(-1,1)+1)/2
    n = min(8, len(preds))
    grid = torch.cat([denorm(torch.cat(inps[:n])),
                      denorm(torch.cat(preds[:n])),
                      denorm(torch.cat(tgts[:n]))], dim=-1)
    save_image(grid, os.path.join(args.output_dir,"grid_"+tag+".png"),
               nrow=1, padding=4, pad_value=1.0)
    print("Outputs saved:", args.output_dir)
    return summary

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--data_root",required=True); p.add_argument("--model_AB",required=True)
    p.add_argument("--model_BA",required=True);  p.add_argument("--direction",default="AB")
    p.add_argument("--split",default="test");    p.add_argument("--image_size",type=int,default=256)
    p.add_argument("--ssim_threshold",type=float,default=0.70)
    p.add_argument("--max_iterations",type=int,default=3)
    p.add_argument("--output_dir",default="./results")
    evaluate(p.parse_args())


In [ ]:
import os
files = ["geoagent_dataset.py","geoagent_model.py","geoagent_train.py",
         "geoagent_agent.py","geoagent_evaluate.py"]
all_ok = True
for fname in files:
    size = os.path.getsize(fname) if os.path.exists(fname) else 0
    ok   = size > 100
    print(("OK" if ok else "MISSING"), fname, "({} bytes)".format(size))
    if not ok: all_ok = False
print("All files ready." if all_ok else "Some files missing — re-run Cell 2.")


## 3. Extract Datasets

Extracts your local zip files from `D:/First year/Papers/GeoAgent/`.
Run once — skips if already extracted.

In [ ]:
import os, zipfile

# ── YOUR LOCAL PATHS ────────────────────────────────────────
BASE_DIR = r"D:\First year\Papers\GeoAgent"

ZIPS = [
    ("24Paired_Porosity_Facies_split",  "24Paired_Porosity_Facies_split.zip"),
    ("Paired_PERM_Facies_split",        "Paired_PERM_Facies_split0.zip"),
]

for folder_name, zip_name in ZIPS:
    folder_path = os.path.join(BASE_DIR, folder_name)
    zip_path    = os.path.join(BASE_DIR, zip_name)

    if os.path.isdir(folder_path):
        splits = {}
        for s in ["train","val","test"]:
            sp = os.path.join(folder_path, s)
            if os.path.exists(sp): splits[s] = len(os.listdir(sp))
        print("Already extracted:", folder_name, splits)
        continue

    if os.path.exists(zip_path):
        print("Extracting", zip_name, "...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(BASE_DIR)
        print("  Done:", folder_path)
    else:
        print("ZIP NOT FOUND:", zip_path)
        print("  -> Check filename matches exactly (case-sensitive)")

print()
print("Dataset structure check:")
for folder_name, _ in ZIPS:
    path = os.path.join(BASE_DIR, folder_name)
    if os.path.isdir(path):
        splits = {}
        for s in ["train","val","test"]:
            sp = os.path.join(path, s)
            if os.path.exists(sp): splits[s] = len(os.listdir(sp))
        print(" ", folder_name, splits)
    else:
        print("  MISSING:", folder_name)


## 4. Configure Paths

In [ ]:
import os, sys

# ══════════════════════════════════════════════════════════════════════════════
# MASTER CONFIG — edit this cell only
# ══════════════════════════════════════════════════════════════════════════════
BASE_DIR = r"D:\First year\Papers\GeoAgent"

# ── Dataset paths ─────────────────────────────────────────────────────────────
# v1 datasets (already extracted)
DATASET_POR_FAC  = os.path.join(BASE_DIR, "24Paired_Porosity_Facies_split")
DATASET_FAC_PERM = os.path.join(BASE_DIR, "Paired_PERM_Facies_split")

# v3 datasets (same folders, both 2D and 3D use same paired PNGs)
DATA_DIR             = BASE_DIR                          # alias used in later sections
DATASET_2D_POR_FAC   = DATASET_POR_FAC
DATASET_3D_POR_FAC   = DATASET_POR_FAC
DATASET_2D_FAC_PERM  = DATASET_FAC_PERM
DATASET_3D_FAC_PERM  = DATASET_FAC_PERM

# ── Output dirs ───────────────────────────────────────────────────────────────
CHECKPOINTS_DIR = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR     = os.path.join(BASE_DIR, "results")
LOGS_DIR        = os.path.join(BASE_DIR, "logs")
for d in [CHECKPOINTS_DIR, RESULTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Model / training hyperparams ──────────────────────────────────────────────
IMAGE_SIZE   = 256
BATCH_SIZE   = 8
NGF          = 128          # feature channels (ngf=128 = v2 stronger model)
EPOCHS_3D    = 300
EPOCHS_2D    = 300

# ── Agent / evaluation ────────────────────────────────────────────────────────
SSIM_THRESH  = 0.70
MAX_RETRIES  = 3
MC_SAMPLES   = 20

# ── LLM ──────────────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
LLM_MODEL         = "claude-sonnet-5"
LLM_ENABLED       = bool(ANTHROPIC_API_KEY)

# ── Quick checks ──────────────────────────────────────────────────────────────
def check_dataset(path, name):
    if not os.path.isdir(path):
        print("MISSING:", name, "->", path); return False
    splits = {s: len(os.listdir(os.path.join(path, s)))
              for s in ["train","val","test"] if os.path.isdir(os.path.join(path, s))}
    print("OK:", name, splits); return True

check_dataset(DATASET_POR_FAC,  "PorFac  (Porosity <-> Facies)")
check_dataset(DATASET_FAC_PERM, "FacPerm (Facies <-> Permeability)")
print()
print("BASE_DIR       :", BASE_DIR)
print("CHECKPOINTS_DIR:", CHECKPOINTS_DIR)
print("RESULTS_DIR    :", RESULTS_DIR)
print("NGF={} | IMAGE_SIZE={} | LLM_ENABLED={}".format(NGF, IMAGE_SIZE, LLM_ENABLED))


## 5. Preview Dataset Pairs

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os, random

def preview_dataset(root, title, n=4):
    train_dir = os.path.join(root, "train")
    if not os.path.exists(train_dir):
        print("Not found:", train_dir); return
    fnames = sorted(os.listdir(train_dir))
    sample = random.sample(fnames, min(n, len(fnames)))
    fig, axes = plt.subplots(n, 3, figsize=(14, 3*n))
    if n == 1: axes = [axes]
    fig.suptitle(title, fontsize=13, fontweight="bold")
    for ax, lbl in zip(axes[0], ["Property A (Left)", "Property B (Right)", "Full Pair"]):
        ax.set_title(lbl, fontsize=9)
    for row, fname in enumerate(sample):
        img = Image.open(os.path.join(train_dir, fname)).convert("RGB")
        w, h = img.size; mid = w // 2
        axes[row][0].imshow(img.crop((0,0,mid,h)));  axes[row][0].axis("off")
        axes[row][1].imshow(img.crop((mid,0,w,h))); axes[row][1].axis("off")
        axes[row][2].imshow(img);                    axes[row][2].axis("off")
        axes[row][0].set_ylabel(fname[:22], fontsize=7, rotation=0, labelpad=90)
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, "preview_" + title[:20].replace(" ","_") + ".png")
    plt.savefig(out, dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

preview_dataset(DATASET_POR_FAC,  "Dataset 1: Porosity vs Facies")
preview_dataset(DATASET_FAC_PERM, "Dataset 2: Facies vs Permeability")


## 6. Verify Dataset Loader

In [ ]:
import torch, sys
for mod in ["geoagent_dataset"]:
    if mod in sys.modules: del sys.modules[mod]

from geoagent_dataset import build_dataloaders

loaders = build_dataloaders(
    data_root=DATASET_POR_FAC,
    direction="AB",
    image_size=IMAGE_SIZE,
    batch_size=4,
    num_workers=0,
)
batch = next(iter(loaders["train"]))
print("Input  (Porosity):", tuple(batch["input"].shape),
      "  range [{:.2f}, {:.2f}]".format(float(batch["input"].min()), float(batch["input"].max())))
print("Target (Facies):  ", tuple(batch["target"].shape),
      "  range [{:.2f}, {:.2f}]".format(float(batch["target"].min()), float(batch["target"].max())))
print("Sample coords:", batch["coords"])
print("Train batches:", len(loaders["train"]),
      "| Val:", len(loaders["val"]),
      "| Test:", len(loaders["test"]))
print("Loader OK.")


In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

## 7. Initialize Model & Smoke-test

*Confirms `inplace=False` fix — should run without RuntimeError.*

In [ ]:
import torch, sys
for mod in ["geoagent_model"]:
    if mod in sys.modules: del sys.modules[mod]

from geoagent_model import Pix2PixModel, count_params

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = Pix2PixModel(device=device)
print(count_params(model))

torch.manual_seed(42)
x = torch.randn(2, 3, 256, 256)
y = torch.randn(2, 3, 256, 256)
losses = model.train_step(x, y)
print("Losses:", {k: round(v,4) for k,v in losses.items()})

if torch.cuda.is_available():
    mem = torch.cuda.max_memory_allocated() / 1e9
    print("GPU memory (batch=2): {:.2f} GB".format(mem))

del model
print("Model OK — no inplace errors.")


## 8. Training

| Step | Direction | Dataset | Meaning | Time (GPU) |
|------|-----------|---------|---------|------------|
| 8a | AB | Por-Fac | Porosity → Facies | ~3.5 hrs |
| 8b | BA | Por-Fac | Facies → Porosity | ~3.5 hrs |
| 8c | AB | Fac-Perm | Facies → Permeability | ~3.5 hrs |

**If training is interrupted:** add `resume=` pointing to the last `.pth` file to continue.

In [ ]:
import subprocess, sys, os

def run_training(data_root, direction, tag, epochs=200, batch_size=4, resume=None):
    ckpt_dir = os.path.join(CHECKPOINTS_DIR, tag)
    os.makedirs(ckpt_dir, exist_ok=True)
    cmd = [
        sys.executable, "geoagent_train.py",
        "--data_root",   data_root,
        "--direction",   direction,
        "--epochs",      str(epochs),
        "--batch_size",  str(batch_size),
        "--image_size",  str(IMAGE_SIZE),
        "--save_dir",    ckpt_dir,
        "--num_workers", "0",
    ]
    if resume:
        cmd += ["--resume", resume]
    print("=" * 58)
    print("  Tag:", tag, "| dir:", direction, "| epochs:", epochs)
    print("  Checkpoints ->", ckpt_dir)
    print("=" * 58)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            cwd=os.path.dirname(os.path.abspath("geoagent_train.py")))
    for line in proc.stdout: print(line, end="")
    proc.wait()
    status = "DONE" if proc.returncode == 0 else "ERROR ({})".format(proc.returncode)
    print("Training", status, "->", tag)
    return ckpt_dir

# ── 8a: Porosity -> Facies ──────────────────────────────────────────────────
ckpt_AB_pf = run_training(DATASET_POR_FAC, "AB", "PorFac_AB")
# To resume: run_training(..., resume=os.path.join(CHECKPOINTS_DIR,"PorFac_AB","pix2pix_AB_ep0190.pth"))


In [ ]:
# ── 8b: Facies -> Porosity ──────────────────────────────────────────────────
ckpt_BA_pf = run_training(DATASET_POR_FAC, "BA", "PorFac_BA")


In [ ]:
# ── 8c: Facies -> Permeability ─────────────────────────────────────────────
#Run after 8a & 8b complete.
ckpt_AB_fp = run_training(DATASET_FAC_PERM, "AB", "FacPerm_AB")
print("Uncomment above to train Facies -> Permeability.")


In [ ]:
# ── 8d: Permeability -> Facies (BA) ────────────────────────────────────────
ckpt_BA_fp = run_training(DATASET_FAC_PERM, "BA", "FacPerm_BA")

## 9. Training Loss Curves

In [ ]:
import json, os
import matplotlib.pyplot as plt

def plot_log(log_path, title):
    if not os.path.exists(log_path):
        print("Log not found:", log_path); return
    with open(log_path) as f: h = json.load(f)
    ep   = [x["epoch"]      for x in h]
    gT   = [x["loss_G"]     for x in h]
    gGAN = [x["loss_G_gan"] for x in h]
    gL1  = [x["loss_G_l1"]  for x in h]
    dL   = [x["loss_D"]     for x in h]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(title, fontsize=12, fontweight="bold")
    ax1.plot(ep, gT,   label="G total",  lw=1.5)
    ax1.plot(ep, gGAN, label="G GAN",    lw=1, ls="--")
    ax1.plot(ep, gL1,  label="G L1x100", lw=1, ls=":")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title("Generator"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(ep, dL, color="tomato", lw=1.5)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
    ax2.set_title("Discriminator"); ax2.grid(alpha=0.3)
    plt.tight_layout()
    out = log_path.replace(".json","_curve.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)
    

plot_log(os.path.join(CHECKPOINTS_DIR,"PorFac_AB","training_log_AB.json"),
         "Porosity to Facies (AB)")
plot_log(os.path.join(CHECKPOINTS_DIR,"PorFac_BA","training_log_BA.json"),
         "Facies to Porosity (BA)")


In [ ]:
import json, os
import matplotlib.pyplot as plt

def plot_log(log_path, title):
    if not os.path.exists(log_path):
        print("Log not found:", log_path); return
    with open(log_path) as f: h = json.load(f)
    ep   = [x["epoch"]      for x in h]
    gT   = [x["loss_G"]     for x in h]
    gGAN = [x["loss_G_gan"] for x in h]
    gL1  = [x["loss_G_l1"]  for x in h]
    dL   = [x["loss_D"]     for x in h]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(title, fontsize=12, fontweight="bold")
    ax1.plot(ep, gT,   label="G total",  lw=1.5)
    ax1.plot(ep, gGAN, label="G GAN",    lw=1, ls="--")
    ax1.plot(ep, gL1,  label="G L1x100", lw=1, ls=":")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title("Generator"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(ep, dL, color="tomato", lw=1.5)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
    ax2.set_title("Discriminator"); ax2.grid(alpha=0.3)
    plt.tight_layout()
    out = log_path.replace(".json","_curve.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out)
    

plot_log(os.path.join(CHECKPOINTS_DIR,"FacPerm_AB","training_log_AB.json"),
         "Permeability to Facies (AB)")
plot_log(os.path.join(CHECKPOINTS_DIR,"FacPerm_BA","training_log_BA.json"),
         "Facies to Permeability (BA)")


In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    "font.family":     "DejaVu Sans",
    "font.size":       11,
    "axes.linewidth":  1.2,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi":      150,
})

def smooth(vals, w=7):
    """Moving average smoothing for cleaner curves."""
    if len(vals) < w: return vals
    kernel = np.ones(w)/w
    return np.convolve(vals, kernel, mode="same").tolist()

def plot_all_logs(configs, out_path, dpi=300):
    """
    configs: list of (log_path, label, color)
    Plots G total + D loss for all configs on same axes.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    for log_path, label, color in configs:
        if not os.path.exists(log_path):
            print("Missing:", log_path); continue
        with open(log_path) as f: h = json.load(f)
        ep   = [x["epoch"]    for x in h]
        gT   = smooth([x["loss_G"]   for x in h])
        dL   = smooth([x["loss_D"]   for x in h])
        ax1.plot(ep, gT, color=color, lw=1.8, label=label)
        ax2.plot(ep, dL, color=color, lw=1.8, label=label)

    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Generator Loss (G total)", fontsize=12)
    ax1.set_title("Generator Training Dynamics", fontsize=13, fontweight="bold")
    ax1.legend(fontsize=10, framealpha=0.9)
    ax1.grid(alpha=0.25, linestyle="--")
    ax1.yaxis.set_minor_locator(ticker.AutoMinorLocator())

    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("Discriminator Loss", fontsize=12)
    ax2.set_title("Discriminator Training Dynamics", fontsize=13, fontweight="bold")
    ax2.legend(fontsize=10, framealpha=0.9)
    ax2.grid(alpha=0.25, linestyle="--")
    ax2.yaxis.set_minor_locator(ticker.AutoMinorLocator())

    # Healthy D-loss reference band
    ax2.axhspan(0.25, 0.45, alpha=0.08, color="green", label="Healthy D range")
    ax2.text(5, 0.35, "Healthy D range", fontsize=8, color="green", alpha=0.7)

    plt.suptitle("GeoAgent-LLM Training Curves — All Directions",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    print("Saved:", out_path)

def plot_single_detail(log_path, title, out_path, dpi=300):
    """Detailed 4-panel plot for one direction: G total, G GAN, G L1, D."""
    if not os.path.exists(log_path):
        print("Missing:", log_path); return
    with open(log_path) as f: h = json.load(f)
    ep   = [x["epoch"]        for x in h]
    gT   = smooth([x["loss_G"]        for x in h])
    gGAN = smooth([x["loss_G_gan"]    for x in h])
    gL1  = smooth([x["loss_G_l1"]     for x in h])
    gPRC = smooth([x.get("loss_G_perc",0) for x in h])
    gSSIM= smooth([x.get("loss_G_ssim",0) for x in h])
    dL   = smooth([x["loss_D"]        for x in h])

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    panels = [
        (axes[0,0], gT,    "G Total Loss",       "#1f77b4", None),
        (axes[0,1], gGAN,  "G GAN Loss",         "#ff7f0e", None),
        (axes[0,2], gL1,   "G L1 Loss",          "#2ca02c", None),
        (axes[1,0], gPRC,  "G Perceptual Loss",  "#9467bd", None),
        (axes[1,1], gSSIM, "G SSIM Loss",        "#8c564b", None),
        (axes[1,2], dL,    "Discriminator Loss", "#d62728",
         (0.25, 0.45, "Healthy D range")),
    ]
    for ax, vals, label, color, band in panels:
        ax.plot(ep, vals, color=color, lw=1.8)
        if band:
            lo, hi, txt = band
            ax.axhspan(lo, hi, alpha=0.1, color="green")
            ax.text(ep[0]+2, (lo+hi)/2, txt, fontsize=8, color="green")
        ax.set_title(label, fontsize=11, fontweight="bold")
        ax.set_xlabel("Epoch", fontsize=10)
        ax.set_ylabel("Loss", fontsize=10)
        ax.grid(alpha=0.25, linestyle="--")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.suptitle(title, fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    print("Saved:", out_path)

# ── Plot all 4 directions together (paper Figure) ────────────────────────────
CKPT = CHECKPOINTS_DIR
configs_all = [
    (os.path.join(CKPT,"PorFac_AB","training_log_AB.json"), "Por→Fac (AB)", "#1f77b4"),
    (os.path.join(CKPT,"PorFac_BA","training_log_BA.json"), "Fac→Por (BA)", "#ff7f0e"),
    (os.path.join(CKPT,"FacPerm_AB","training_log_AB.json"),"Fac→Perm (AB)","#2ca02c"),
    (os.path.join(CKPT,"FacPerm_BA","training_log_BA.json"),"Perm→Fac (BA)","#9467bd"),
]
plot_all_logs(configs_all,
              os.path.join(RESULTS_DIR,"fig_training_curves_all.png"), dpi=300)

# ── Individual detailed plots for each direction ──────────────────────────────
for tag, direction, title in [
    ("PorFac_AB",  "AB", "GeoAgent-LLM | Porosity → Facies (AB)"),
    ("PorFac_BA",  "BA", "GeoAgent-LLM | Facies → Porosity (BA)"),
    ("FacPerm_AB", "AB", "GeoAgent-LLM | Facies → Permeability (AB)"),
    ("FacPerm_BA", "BA", "GeoAgent-LLM | Permeability → Facies (BA)"),
]:
    log_p = os.path.join(CKPT, tag, "training_log_{}.json".format(direction))
    out_p = os.path.join(RESULTS_DIR, "fig_training_{}.png".format(tag))
    plot_single_detail(log_p, title, out_p, dpi=300)

## 10. GeoAgent Evaluation

In [ ]:
import sys, os, argparse

for mod in ["geoagent_agent","geoagent_evaluate","geoagent_model"]:
    if mod in sys.modules: del sys.modules[mod]

from geoagent_evaluate import evaluate

def make_v1_args(data_root, model_AB, model_BA, direction, tag):
    return argparse.Namespace(
        data_root      = data_root,
        model_AB       = model_AB,
        model_BA       = model_BA,
        direction      = direction,
        split          = "test",
        image_size     = IMAGE_SIZE,
        ssim_threshold = 0.70,
        max_iterations = 3,
        output_dir     = os.path.join(RESULTS_DIR, tag),
    )

# ── PorFac checkpoints ────────────────────────────────────────────────────────
PF_AB = os.path.join(CHECKPOINTS_DIR, "PorFac_AB", "pix2pix_AB_ep0200.pth")
PF_BA = os.path.join(CHECKPOINTS_DIR, "PorFac_BA", "pix2pix_BA_ep0200.pth")

if os.path.exists(PF_AB) and os.path.exists(PF_BA):
    print("=" * 55)
    print("--- Evaluating AB: Porosity -> Facies ---")
    evaluate(make_v1_args(DATASET_POR_FAC, PF_AB, PF_BA, "AB", "PorFac"))
    print()
    print("--- Evaluating BA: Facies -> Porosity ---")
    evaluate(make_v1_args(DATASET_POR_FAC, PF_AB, PF_BA, "BA", "PorFac"))
else:
    print("PorFac checkpoints missing — train first (Cell 8).")
    for p in [PF_AB, PF_BA]: print(" ", "OK" if os.path.exists(p) else "MISSING", p)


In [ ]:
import argparse, os

# ── FacPerm checkpoints ───────────────────────────────────────────────────────
FP_AB = os.path.join(CHECKPOINTS_DIR, "FacPerm_AB", "pix2pix_AB_ep0200.pth")
FP_BA = os.path.join(CHECKPOINTS_DIR, "FacPerm_BA", "pix2pix_BA_ep0200.pth")

def make_v1_fp_args(direction):
    return argparse.Namespace(
        data_root      = DATASET_FAC_PERM,
        model_AB       = FP_AB,
        model_BA       = FP_BA,
        direction      = direction,
        split          = "test",
        image_size     = IMAGE_SIZE,
        ssim_threshold = 0.65,
        max_iterations = 3,
        output_dir     = os.path.join(RESULTS_DIR, "FacPerm_" + direction),
    )

if os.path.exists(FP_AB) and os.path.exists(FP_BA):
    print("=" * 55)
    print("--- Evaluating AB: Facies -> Permeability ---")
    evaluate(make_v1_fp_args("AB"))
    print()
    print("--- Evaluating BA: Permeability -> Facies ---")
    evaluate(make_v1_fp_args("BA"))
else:
    print("FacPerm checkpoints missing — train first (Cell 8c/8d).")
    for p in [FP_AB, FP_BA]: print(" ", "OK" if os.path.exists(p) else "MISSING", p)


## 11. Visual Comparison Grid

*Input | GeoAgent Output | Ground Truth*

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

def show_grid(tag, direction, title=None):
    path = os.path.join(RESULTS_DIR, tag, "grid_" + direction + "_test.png")
    if not os.path.exists(path):
        print("Not found:", path, "(run Cell 10 first)"); return
    img = Image.open(path)
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.imshow(img); ax.axis("off")
    ax.set_title(title or (tag + " | " + direction +
                 "   [Input | GeoAgent Output | Ground Truth]"),
                 fontsize=12, pad=10)
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, tag + "_" + direction + "_vis.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", out)

# PorFac
show_grid("PorFac",     "AB", "Porosity → Facies  [Input | Pred | GT]")
show_grid("PorFac",     "BA", "Facies → Porosity  [Input | Pred | GT]")
# FacPerm
show_grid("FacPerm_AB", "AB", "Facies → Permeability  [Input | Pred | GT]")
show_grid("FacPerm_BA", "BA", "Permeability → Facies  [Input | Pred | GT]")


## 12. Paper-Ready Results Table

In [ ]:
import json, os
import pandas as pd

def load_summary(tag, direction):
    path = os.path.join(RESULTS_DIR, tag, "summary_" + direction + "_test.json")
    if not os.path.exists(path): return None
    with open(path) as f: return json.load(f)

configs = [
    ("PorFac",     "AB", "Porosity → Facies"),
    ("PorFac",     "BA", "Facies → Porosity"),
    ("FacPerm_AB", "AB", "Facies → Permeability"),
    ("FacPerm_BA", "BA", "Permeability → Facies"),
]

rows = []
for tag, direction, label in configs:
    s = load_summary(tag, direction)
    if s is None:
        print("No result yet for:", label, "(run evaluation first)")
        continue
    rows.append({
        "Translation": label,
        "SSIM":    "{:.4f}±{:.4f}".format(s["ssim"]["mean"],      s["ssim"]["std"]),
        "PSNR dB": "{:.2f}±{:.2f}".format(s["psnr_db"]["mean"],  s["psnr_db"]["std"]),
        "PixAcc":  "{:.4f}±{:.4f}".format(s["pixel_acc"]["mean"], s["pixel_acc"]["std"]),
        "Accept%": "{:.1f}%".format(s.get("acceptance_rate", 0)*100),
        "AvgIter": "{:.2f}".format(s["n_iter"]["mean"]),
    })

if rows:
    df = pd.DataFrame(rows)
    print("=== GeoAgent v1 Test Set Results — All Directions ===")
    print()
    print(df.to_string(index=False))
    out = os.path.join(RESULTS_DIR, "paper_table_v1.csv")
    df.to_csv(out, index=False)
    print("\nSaved:", out)


## 13. Single-Image GeoAgent Demo

Full reasoning trace on one test image.

In [ ]:
import sys, os, torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

for mod in ["geoagent_agent","geoagent_model"]:
    if mod in sys.modules: del sys.modules[mod]

from geoagent_agent import GeoAgent, tensor_to_pil

AB_ckpt = os.path.join(CHECKPOINTS_DIR, "PorFac_AB", "pix2pix_AB_ep0200.pth")
BA_ckpt = os.path.join(CHECKPOINTS_DIR, "PorFac_BA", "pix2pix_BA_ep0200.pth")

if not os.path.exists(AB_ckpt):
    print("Train first (Cell 8a). Checkpoint not found:", AB_ckpt)
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent  = GeoAgent(AB_ckpt, BA_ckpt, device=device,
                      ssim_threshold=0.70, max_iterations=3, verbose=True)

    test_dir = os.path.join(DATASET_POR_FAC, "test")
    test_img = os.path.join(test_dir, sorted(os.listdir(test_dir))[0])
    print("Using:", os.path.basename(test_img))

    full = Image.open(test_img).convert("RGB")
    w, h = full.size
    inp_pil = full.crop((0,    0, w//2, h))
    tgt_pil = full.crop((w//2, 0, w,   h))

    tfm = transforms.Compose([
        transforms.Resize((256,256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3,[0.5]*3),
    ])
    inp_t = tfm(inp_pil).unsqueeze(0)
    tgt_t = tfm(tgt_pil).unsqueeze(0)

    print()
    print("--- Running GeoAgent ---")
    result = agent.run(inp_t, tgt_t, direction="AB")
    print()
    print(result.summary())
    print()
    print("Reasoning trace ({} steps):".format(len(result.reasoning_trace)))
    for st in result.reasoning_trace:
        mark = "OK" if st.accepted else ("--" if st.accepted is None else "FAIL")
        print("  Step", st.step, "[" + st.action + "]", mark, "|", st.observation)

    pred_pil = tensor_to_pil(result.predicted_image)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    data = [
        (inp_pil,  "Input (Porosity)"),
        (pred_pil, "GeoAgent Output  SSIM={:.4f}".format(result.ssim_score)),
        (tgt_pil,  "Ground Truth (Facies)"),
    ]
    for ax, (img, title) in zip(axes, data):
        ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis("off")
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, "single_demo.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", out)


## 14. Dataset 2: Facies → Permeability

Dataset: `Paired_PERM_Facies_split0`

Run extraction (Cell 3) first, then run these cells.

In [ ]:
import os
from geoagent_dataset import build_dataloaders

if os.path.isdir(DATASET_FAC_PERM):
    ldr = build_dataloaders(DATASET_FAC_PERM, "AB", IMAGE_SIZE, 2, 0)
    b   = next(iter(ldr["train"]))
    print("Fac-Perm loader OK:", tuple(b["input"].shape))
else:
    print("Dataset not found:", DATASET_FAC_PERM)
    print("Run Cell 3 first to extract Paired_PERM_Facies_split0.zip")


In [ ]:
import sys, os, argparse

# Train Facies -> Permeability (run after Cell 8c finishes)
FP_ckpt = os.path.join(CHECKPOINTS_DIR, "FacPerm_AB", "pix2pix_AB_ep0200.pth")

# Need a BA model for the agent — reuse PorFac_BA or use AB as proxy
BA_ckpt_proxy = os.path.join(CHECKPOINTS_DIR, "PorFac_BA", "pix2pix_BA_ep0200.pth")

for mod in ["geoagent_agent","geoagent_evaluate","geoagent_model"]:
    if mod in sys.modules: del sys.modules[mod]
from geoagent_evaluate import evaluate

if os.path.exists(FP_ckpt) and os.path.exists(BA_ckpt_proxy):
    args = argparse.Namespace(
        data_root=DATASET_FAC_PERM, model_AB=FP_ckpt, model_BA=BA_ckpt_proxy,
        direction="AB", split="test", image_size=IMAGE_SIZE,
        ssim_threshold=0.65, max_iterations=3,
        output_dir=os.path.join(RESULTS_DIR, "FacPerm"),
    )
    print("--- Evaluating: Facies -> Permeability ---")
    summary_fp = evaluate(args)
else:
    print("Train Cell 8c first. Checkpoint expected:", FP_ckpt)


## A. Stronger Model — ngf=128 + Perceptual Loss + 3D Context

Upgrades over base model:
- **ngf=128** (2× feature channels → much sharper textures)
- **VGG16 perceptual loss** (high-frequency geological detail)
- **9-channel 3D input** (center + i-1 + i+1 neighbor slices)
- **Spectral norm** on discriminator (training stability)
- **SSIM loss** component (structural preservation)
- **512×512 output** (high resolution saves)

In [ ]:
%%writefile geoagent_model_v2.py
"""Stronger Pix2Pix v2: ngf=128, perceptual loss, spectral norm. Author: Abdulrahman Al-Fakih"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# ── VGG Perceptual Loss ──────────────────────────────────────────────────────
class VGGPerceptualLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features
        # Use layers: relu1_2 (idx 4), relu2_2 (idx 9), relu3_3 (idx 16)
        self.slice1 = nn.Sequential(*list(vgg.children())[:4]).to(device).eval()
        self.slice2 = nn.Sequential(*list(vgg.children())[:9]).to(device).eval()
        self.slice3 = nn.Sequential(*list(vgg.children())[:16]).to(device).eval()
        for p in self.parameters(): p.requires_grad = False
        self.weights = [1.0, 0.75, 0.5]
    def forward(self, pred, target):
        # Convert [-1,1] -> [0,1] for VGG
        p = (pred.clamp(-1,1)+1)/2
        t = (target.clamp(-1,1)+1)/2
        # If 9-channel input, use only center 3 channels
        if p.shape[1] == 9: p = p[:,3:6]
        if t.shape[1] == 9: t = t[:,3:6]
        loss = 0.0
        for sl, w in zip([self.slice1, self.slice2, self.slice3], self.weights):
            loss += w * F.l1_loss(sl(p), sl(t))
        return loss

# ── SSIM Loss ────────────────────────────────────────────────────────────────
def ssim_loss(pred, target, window_size=11):
    p = (pred.clamp(-1,1)+1)/2
    t = (target.clamp(-1,1)+1)/2
    if p.shape[1] > 3: p = p[:,3:6]; t = t[:,3:6]
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(p, window_size, 1, window_size//2)
    mu_t = F.avg_pool2d(t, window_size, 1, window_size//2)
    mu_p2, mu_t2, mu_pt = mu_p**2, mu_t**2, mu_p*mu_t
    sig_p = F.avg_pool2d(p*p, window_size, 1, window_size//2) - mu_p2
    sig_t = F.avg_pool2d(t*t, window_size, 1, window_size//2) - mu_t2
    sig_pt= F.avg_pool2d(p*t, window_size, 1, window_size//2) - mu_pt
    num   = (2*mu_pt+C1)*(2*sig_pt+C2)
    den   = (mu_p2+mu_t2+C1)*(sig_p+sig_t+C2)
    return 1.0 - (num/den).mean()

# ── Building Block ───────────────────────────────────────────────────────────
class ConvNormAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=4, stride=2, padding=1,
                 norm="batch", act="leaky", transpose=False, spectral=False):
        super().__init__()
        bias = (norm == "none")
        if transpose:
            conv = nn.ConvTranspose2d(in_ch, out_ch, kernel, stride, padding, bias=bias)
        else:
            conv = nn.Conv2d(in_ch, out_ch, kernel, stride, padding, bias=bias)
        if spectral: conv = nn.utils.spectral_norm(conv)
        layers = [conv]
        if norm == "batch":      layers.append(nn.BatchNorm2d(out_ch))
        elif norm == "instance": layers.append(nn.InstanceNorm2d(out_ch, affine=True))
        if act == "leaky": layers.append(nn.LeakyReLU(0.2, inplace=False))
        elif act == "relu": layers.append(nn.ReLU(inplace=False))
        elif act == "tanh": layers.append(nn.Tanh())
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

# ── U-Net Generator (ngf=128, 9-channel 3D input) ───────────────────────────
class UNetGenerator(nn.Module):
    def __init__(self, in_ch=9, out_ch=3, ngf=128, norm="batch"):
        super().__init__()
        self.e1 = nn.Conv2d(in_ch, ngf, 4, 2, 1, bias=False)
        self.e2 = ConvNormAct(ngf,    ngf*2,  norm=norm)
        self.e3 = ConvNormAct(ngf*2,  ngf*4,  norm=norm)
        self.e4 = ConvNormAct(ngf*4,  ngf*8,  norm=norm)
        self.e5 = ConvNormAct(ngf*8,  ngf*8,  norm=norm)
        self.e6 = ConvNormAct(ngf*8,  ngf*8,  norm=norm)
        self.e7 = ConvNormAct(ngf*8,  ngf*8,  norm=norm)
        self.e8 = nn.Sequential(
            nn.LeakyReLU(0.2, inplace=False),
            nn.Conv2d(ngf*8, ngf*8, 4, 2, 1, bias=False))
        self.d8 = self._dec(ngf*8,   ngf*8,  drop=True,  norm=norm)
        self.d7 = self._dec(ngf*16,  ngf*8,  drop=True,  norm=norm)
        self.d6 = self._dec(ngf*16,  ngf*8,  drop=True,  norm=norm)
        self.d5 = self._dec(ngf*16,  ngf*8,  drop=False, norm=norm)
        self.d4 = self._dec(ngf*16,  ngf*4,  drop=False, norm=norm)
        self.d3 = self._dec(ngf*8,   ngf*2,  drop=False, norm=norm)
        self.d2 = self._dec(ngf*4,   ngf,    drop=False, norm=norm)
        self.d1 = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.ConvTranspose2d(ngf*2, out_ch, 4, 2, 1, bias=False),
            nn.Tanh())
    @staticmethod
    def _dec(in_ch, out_ch, drop=False, norm="batch"):
        layers = [nn.ReLU(inplace=False),
                  nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if norm == "batch":      layers.append(nn.BatchNorm2d(out_ch))
        elif norm == "instance": layers.append(nn.InstanceNorm2d(out_ch, affine=True))
        if drop: layers.append(nn.Dropout(0.5))
        return nn.Sequential(*layers)
    def forward(self, x):
        e1=self.e1(x); e2=self.e2(e1); e3=self.e3(e2); e4=self.e4(e3)
        e5=self.e5(e4); e6=self.e6(e5); e7=self.e7(e6); e8=self.e8(e7)
        d8=self.d8(e8)
        d7=self.d7(torch.cat([d8,e7],1))
        d6=self.d6(torch.cat([d7,e6],1))
        d5=self.d5(torch.cat([d6,e5],1))
        d4=self.d4(torch.cat([d5,e4],1))
        d3=self.d3(torch.cat([d4,e3],1))
        d2=self.d2(torch.cat([d3,e2],1))
        return self.d1(torch.cat([d2,e1],1))

# ── PatchGAN Discriminator with Spectral Norm ────────────────────────────────
class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_ch=12, ndf=64, n_layers=3, norm="instance"):
        super().__init__()
        # in_ch = 9 (input 3D) + 3 (output) = 12 for AB
        seq = [nn.utils.spectral_norm(nn.Conv2d(in_ch,ndf,4,2,1,bias=False)),
               nn.LeakyReLU(0.2,inplace=False)]
        ch = ndf
        for n in range(1, n_layers):
            ch_prev, ch = ch, min(ch*2, 512)
            stride = 1 if n == n_layers-1 else 2
            seq += [nn.utils.spectral_norm(nn.Conv2d(ch_prev,ch,4,stride,1,bias=False)),
                    nn.InstanceNorm2d(ch, affine=True),
                    nn.LeakyReLU(0.2,inplace=False)]
        ch2 = min(ch*2, 512)
        seq += [nn.utils.spectral_norm(nn.Conv2d(ch,ch2,4,1,1,bias=False)),
                nn.InstanceNorm2d(ch2, affine=True),
                nn.LeakyReLU(0.2,inplace=False),
                nn.Conv2d(ch2,1,4,1,1)]
        self.model = nn.Sequential(*seq)
    def forward(self, inp, tgt):
        return self.model(torch.cat([inp,tgt],1))

# ── Full Model ────────────────────────────────────────────────────────────────
class Pix2PixModelV2:
    def __init__(self, device, in_ch=9, out_ch=3, ngf=128, ndf=64,
                 lambda_l1=100.0, lambda_perc=10.0, lambda_ssim=5.0,
                 norm="batch", lr=1e-4, beta1=0.5):
        self.device       = device
        self.lambda_l1    = lambda_l1
        self.lambda_perc  = lambda_perc
        self.lambda_ssim  = lambda_ssim
        disc_in = in_ch + out_ch
        self.G  = UNetGenerator(in_ch, out_ch, ngf, norm).to(device)
        self.D  = PatchGANDiscriminator(disc_in, ndf, norm="instance").to(device)
        self.perc_loss = VGGPerceptualLoss(device)
        self.cGAN = nn.BCEWithLogitsLoss()
        self.cL1  = nn.L1Loss()
        self.oG = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(beta1,0.999))
        self.oD = torch.optim.Adam(self.D.parameters(), lr=lr*0.5, betas=(beta1,0.999))
        self.sG = torch.optim.lr_scheduler.CosineAnnealingLR(self.oG, T_max=300, eta_min=1e-6)
        self.sD = torch.optim.lr_scheduler.CosineAnnealingLR(self.oD, T_max=300, eta_min=1e-7)
        self._init_weights()
    def _init_weights(self):
        for m in list(self.G.modules())+list(self.D.modules()):
            if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d)):
                if not hasattr(m,"weight_orig"):  # skip spectral norm wrapped
                    nn.init.normal_(m.weight,0.0,0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m,(nn.BatchNorm2d,nn.InstanceNorm2d)):
                if m.weight is not None: nn.init.normal_(m.weight,1.0,0.02)
                if m.bias  is not None: nn.init.zeros_(m.bias)
    def _rg(self, net, flag):
        for p in net.parameters(): p.requires_grad = flag
    def train_step(self, rA, rB):
        rA, rB = rA.to(self.device), rB.to(self.device)
        fB = self.G(rA)
        # Discriminator
        self._rg(self.D, True); self.oD.zero_grad()
        pf = self.D(rA, fB.detach())
        pr = self.D(rA, rB)
        lD = 0.5*(self.cGAN(pf,torch.zeros_like(pf))+self.cGAN(pr,torch.ones_like(pr)))
        lD.backward(); self.oD.step()
        # Generator
        self._rg(self.D, False); self.oG.zero_grad()
        pg   = self.D(rA, fB)
        lGg  = self.cGAN(pg, torch.ones_like(pg))
        lGl  = self.cL1(fB, rB) * self.lambda_l1
        lGp  = self.perc_loss(fB, rB) * self.lambda_perc
        lGs  = ssim_loss(fB, rB) * self.lambda_ssim
        lG   = lGg + lGl + lGp + lGs
        lG.backward(); self.oG.step()
        return {"loss_G":lG.item(),"loss_G_gan":lGg.item(),
                "loss_G_l1":lGl.item(),"loss_G_perc":lGp.item(),
                "loss_G_ssim":lGs.item(),"loss_D":lD.item()}
    @torch.no_grad()
    def predict(self, x):
        self.G.eval()
        return self.G(x.to(self.device))
    def step_schedulers(self): self.sG.step(); self.sD.step()
    def save(self, path):
        torch.save({"G":self.G.state_dict(),"D":self.D.state_dict()}, path)
        print("Saved:", path)
    def load(self, path):
        c = torch.load(path, map_location=self.device)
        self.G.load_state_dict(c["G"]); self.D.load_state_dict(c["D"])
        print("Loaded:", path)
    def param_count(self):
        gp = sum(p.numel() for p in self.G.parameters()) / 1e6
        dp = sum(p.numel() for p in self.D.parameters()) / 1e6
        return "G: {:.1f}M  |  D: {:.1f}M".format(gp, dp)


## B. Train V2 — All Directions, All Properties

Trains 4 models:
- PorFac AB (Porosity → Facies) 3D-context
- PorFac BA (Facies → Porosity) 3D-context
- FacPerm AB (Facies → Permeability) 3D-context
- FacPerm BA (Permeability → Facies) 3D-context

Each ~2 hrs on RTX 4500 Ada with batch=8, ngf=128.

In [ ]:
%%writefile geoagent_train_v2.py
"""Training v2 — supports 2D (in_ch=3) and 3D (in_ch=9). Author: Abdulrahman Al-Fakih"""
import os, argparse, time, json
import torch
from torchvision.utils import save_image
from geoagent_model_v2 import Pix2PixModelV2

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data_root",   required=True)
    p.add_argument("--direction",   default="AB", choices=["AB","BA"])
    p.add_argument("--in_ch",       type=int,   default=9)   # 3=2D, 9=3D  ← ADD THIS
    p.add_argument("--epochs",      type=int,   default=300)
    p.add_argument("--batch_size",  type=int,   default=8)
    p.add_argument("--image_size",  type=int,   default=256)
    p.add_argument("--ngf",         type=int,   default=128)
    p.add_argument("--lambda_l1",   type=float, default=100.0)
    p.add_argument("--lambda_perc", type=float, default=10.0)
    p.add_argument("--lambda_ssim", type=float, default=5.0)
    p.add_argument("--lr",          type=float, default=1e-4)
    p.add_argument("--save_dir",    required=True)
    p.add_argument("--save_freq",   type=int,   default=10)
    p.add_argument("--vis_freq",    type=int,   default=5)
    p.add_argument("--num_workers", type=int,   default=0)
    p.add_argument("--resume",      default=None)
    return p.parse_args()

def denorm(t): return (t.clamp(-1,1)+1)/2

def save_samples(model, loader, epoch, out_dir, n=4):
    os.makedirs(out_dir, exist_ok=True)
    batch = next(iter(loader))
    inp = batch["input"][:n]; tgt = batch["target"][:n]
    gen = model.predict(inp).cpu()
    inp_disp = inp[:,3:6] if inp.shape[1]==9 else inp
    grid = torch.cat([denorm(inp_disp), denorm(gen), denorm(tgt)], dim=-1)
    save_image(grid, os.path.join(out_dir, "epoch_{:04d}.png".format(epoch)), nrow=1)

def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # ── Select loader based on in_ch ──────────────────────────────────────────
    if args.in_ch == 9:
        from geoagent_dataset_3d import build_dataloaders_3d as build_loaders
    else:
        from geoagent_dataset_2d import build_dataloaders_2d as build_loaders
    print("V2 Training | dir:", args.direction, "| in_ch:", args.in_ch,
          "| device:", str(device), "| epochs:", args.epochs)
    loaders = build_loaders(args.data_root, args.direction,
                            args.image_size, args.batch_size, args.num_workers)
    model = Pix2PixModelV2(device, in_ch=args.in_ch, out_ch=3,   # ← in_ch passed correctly
                           ngf=args.ngf,
                           lambda_l1=args.lambda_l1,
                           lambda_perc=args.lambda_perc,
                           lambda_ssim=args.lambda_ssim,
                           lr=args.lr)
    print(model.param_count())
    start_epoch = 1
    if args.resume and os.path.exists(args.resume):
        model.load(args.resume)
        try:
            start_epoch = int(os.path.basename(args.resume).split("ep")[1].split(".")[0])+1
        except Exception: start_epoch = 1
    os.makedirs(args.save_dir, exist_ok=True)
    vis_dir  = os.path.join(args.save_dir, "samples")
    log_path = os.path.join(args.save_dir, "log_" + args.direction + ".json")
    history  = []
    for epoch in range(start_epoch, args.epochs+1):
        model.G.train(); model.D.train()
        totals = {"loss_G":0,"loss_G_gan":0,"loss_G_l1":0,
                  "loss_G_perc":0,"loss_G_ssim":0,"loss_D":0}
        t0 = time.time()
        for batch in loaders["train"]:
            ls = model.train_step(batch["input"], batch["target"])
            for k in totals: totals[k] += ls[k]
        n = len(loaders["train"])
        avg = {k: round(v/n,4) for k,v in totals.items()}
        elapsed = round(time.time()-t0, 1)
        print("Ep[{:3d}] G={:.3f} L1={:.2f} Perc={:.3f} SSIM={:.3f} D={:.3f} ({:.0f}s)".format(
              epoch, avg["loss_G"], avg["loss_G_l1"], avg["loss_G_perc"],
              avg["loss_G_ssim"], avg["loss_D"], elapsed))
        model.step_schedulers()
        history.append({"epoch":epoch,"elapsed_s":elapsed,**avg})
        if epoch % args.vis_freq == 0:
            save_samples(model, loaders["val"], epoch, vis_dir)
        if epoch % args.save_freq == 0 or epoch == args.epochs:
            name = "v2_{}_ep{:04d}.pth".format(args.direction, epoch)
            model.save(os.path.join(args.save_dir, name))
        with open(log_path,"w") as f: json.dump(history, f, indent=2)
    print("Done. Saved to:", args.save_dir)

if __name__ == "__main__":
    train(parse_args())

In [ ]:
import subprocess, sys, os

_NGF = NGF  # from master config cell 15

def run_v2(data_root, direction, tag, in_ch=9, epochs=300, batch=8, resume=None):
    """Launch geoagent_train_v2.py as subprocess. Returns checkpoint dir."""
    ckpt_dir = os.path.join(CHECKPOINTS_DIR, tag)
    os.makedirs(ckpt_dir, exist_ok=True)
    cmd = [
        sys.executable, "geoagent_train_v2.py",
        "--data_root",  data_root,
        "--direction",  direction,
        "--in_ch",      str(in_ch),
        "--epochs",     str(epochs),
        "--batch_size", str(batch),
        "--image_size", str(IMAGE_SIZE),
        "--save_dir",   ckpt_dir,
        "--num_workers","0",
        "--ngf",        str(_NGF),
    ]
    if resume: cmd += ["--resume", resume]
    print("=" * 65)
    print("  run_v2 | tag:{} | dir:{} | in_ch:{} | epochs:{}".format(
          tag, direction, in_ch, epochs))
    print("=" * 65)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout: print(line, end="")
    proc.wait()
    status = "DONE" if proc.returncode == 0 else "ERROR({})".format(proc.returncode)
    print(status, "->", tag)
    return ckpt_dir

# ── All 8 training jobs — uncomment one at a time ─────────────────────────────
# 3D (in_ch=9)  ← main contribution
# run_v2(DATASET_3D_POR_FAC,  "AB", "3D_PorFac_AB",  in_ch=9, epochs=300)
# run_v2(DATASET_3D_POR_FAC,  "BA", "3D_PorFac_BA",  in_ch=9, epochs=300)
# run_v2(DATASET_3D_FAC_PERM, "AB", "3D_FacPerm_AB", in_ch=9, epochs=300)
# run_v2(DATASET_3D_FAC_PERM, "BA", "3D_FacPerm_BA", in_ch=9, epochs=300)

# 2D (in_ch=3)  ← ablation baseline
# run_v2(DATASET_2D_POR_FAC,  "AB", "2D_PorFac_AB",  in_ch=3, epochs=300)
# run_v2(DATASET_2D_POR_FAC,  "BA", "2D_PorFac_BA",  in_ch=3, epochs=300)
# run_v2(DATASET_2D_FAC_PERM, "AB", "2D_FacPerm_AB", in_ch=3, epochs=300)
# run_v2(DATASET_2D_FAC_PERM, "BA", "2D_FacPerm_BA", in_ch=3, epochs=300)

print("run_v2() ready — uncomment one block at a time and run.")


In [ ]:
# ── STATUS CHECK — which v2 checkpoints exist? ────────────────────────────────
import os, glob

tags = [
    "3D_PorFac_AB","3D_PorFac_BA","3D_FacPerm_AB","3D_FacPerm_BA",
    "2D_PorFac_AB","2D_PorFac_BA","2D_FacPerm_AB","2D_FacPerm_BA",
]

print("V2 Training Status")
print("=" * 60)
for tag in tags:
    d = os.path.join(CHECKPOINTS_DIR, tag)
    if not os.path.isdir(d):
        print(f"  {tag:<25} NOT STARTED")
        continue
    ckpts = sorted(glob.glob(os.path.join(d, "*.pth")))
    if ckpts:
        last = os.path.basename(ckpts[-1])
        ep_str = last.split("ep")[-1].split(".")[0] if "ep" in last else "?"
        print(f"  {tag:<25} {len(ckpts)} ckpts | last ep {ep_str}")
    else:
        print(f"  {tag:<25} dir exists but no .pth files")


In [ ]:
# Uncomment blocks in Cell 48 and run them one at a time.
# Suggested order (3D first since those are the main contribution):
#   1. run_v2(DATASET_3D_POR_FAC,  "AB", "3D_PorFac_AB",  in_ch=9)
#   2. run_v2(DATASET_3D_POR_FAC,  "BA", "3D_PorFac_BA",  in_ch=9)
#   3. run_v2(DATASET_3D_FAC_PERM, "AB", "3D_FacPerm_AB", in_ch=9)
#   4. run_v2(DATASET_3D_FAC_PERM, "BA", "3D_FacPerm_BA", in_ch=9)
#   5-8: repeat with in_ch=3 for 2D ablation
print("See Cell 48 for run_v2() — uncomment one block at a time.")


## C. High-Resolution Evaluation & Visualization

Saves individual predictions at **300 DPI**, overlays SSIM per image,
produces 3-panel comparison grid, and a spatial quality heatmap across the 3D cube.

In [ ]:
# Run this first to confirm which folder has FacPerm images:
import os, glob

candidates = [
    os.path.join(BASE_DIR, "Paired_PERM_Facies_split"),
    os.path.join(BASE_DIR, "Paired_PERM_Facies_split0"),
    DATASET_FAC_PERM,
]
found = None
for path in candidates:
    if os.path.isdir(path):
        imgs = glob.glob(os.path.join(path, "**/*.png"), recursive=True)
        print(os.path.basename(path), "->", len(imgs), "PNGs  [", path, "]")
        if imgs and found is None:
            found = path
    else:
        print(os.path.basename(path), "-> NOT FOUND")

if found and found != DATASET_FAC_PERM:
    print()
    print("HINT: Update DATASET_FAC_PERM in Cell 15 to:", found)
else:
    print()
    print("DATASET_FAC_PERM OK:", DATASET_FAC_PERM)


In [ ]:
import sys, os, argparse
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from torchvision import transforms

for mod in ["geoagent_agent","geoagent_model_v2"]:
    if mod in sys.modules: del sys.modules[mod]

from geoagent_agent import compute_ssim, compute_pixel_accuracy
from geoagent_model_v2 import Pix2PixModelV2

def denorm(t):
    return (t.clamp(-1,1)+1)/2

def compute_psnr(pred, target):
    p = denorm(pred.cpu()); t = denorm(target.cpu())
    mse = ((p-t)**2).mean().item()
    return 100.0 if mse<1e-10 else float(10*np.log10(1.0/mse))

def tensor_to_img(t):
    arr = (denorm(t).squeeze(0).permute(1,2,0).cpu().numpy()*255)
    return arr.clip(0,255).astype(np.uint8)

def evaluate_hires(model_ckpt, data_root, direction, out_dir,
                   in_ch=9, image_size=256, device=None, prop_label=""):
    """Evaluate one direction: saves 300 DPI panels + CSV + 3D heatmap."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ind_dir = os.path.join(out_dir, "individual")
    os.makedirs(ind_dir, exist_ok=True)

    model = Pix2PixModelV2(device, in_ch=in_ch, out_ch=3, ngf=NGF)
    model.load(model_ckpt)
    model.G.eval()

    if in_ch == 9:
        from geoagent_dataset_3d import build_dataloaders_3d
        loaders = build_dataloaders_3d(data_root, direction, image_size, 1, 0)
    else:
        from geoagent_dataset_2d import build_dataloaders_2d
        loaders = build_dataloaders_2d(data_root, direction, image_size, 1, 0)
    loader = loaders["test"]

    rows, grid_inps, grid_preds, grid_tgts = [], [], [], []
    ssim_map = {}

    print("Evaluating {} test images | {} {} | in_ch={}".format(
          len(loader), prop_label, direction, in_ch))
    for idx, batch in enumerate(loader):
        inp  = batch["input"]
        tgt  = batch["target"]
        path = batch["path"][0]
        # Safe coords — 2D loader has no coords key
        raw    = batch.get("coords", None)
        coords = tuple(int(c[0]) for c in raw) if raw is not None else (0, 0, idx)

        with torch.no_grad():
            pred = model.predict(inp).cpu()

        ssim = compute_ssim(pred, tgt)
        psnr = compute_psnr(pred, tgt)
        acc  = compute_pixel_accuracy(pred, tgt)
        ssim_map[coords] = ssim

        rows.append({
            "idx":idx, "filename":os.path.basename(path),
            "i":coords[0],"j":coords[1],"k":coords[2],
            "ssim":round(ssim,4),"psnr_db":round(psnr,2),"pixel_acc":round(acc,4),
        })

        inp_disp = inp[:,3:6] if in_ch==9 else inp
        inp_img  = tensor_to_img(inp_disp)
        pred_img = tensor_to_img(pred)
        tgt_img  = tensor_to_img(tgt)

        if idx < 30:  # save first 30 individual panels
            fig = plt.figure(figsize=(15, 5))
            gs  = gridspec.GridSpec(1, 3, wspace=0.05)
            for col, (title, img) in enumerate(
                    zip(["Input", "GeoAgent V2 Output (SSIM={:.4f})".format(ssim),
                         "Ground Truth"], [inp_img, pred_img, tgt_img])):
                ax = fig.add_subplot(gs[0, col])
                ax.imshow(img); ax.axis("off")
                ax.set_title(title, fontsize=10, pad=4)
            fig.suptitle("{} | {} | {}  (i={},j={},k={})".format(
                         prop_label, direction, os.path.basename(path),
                         coords[0],coords[1],coords[2]), fontsize=9)
            fig.savefig(os.path.join(ind_dir, "pred_{:05d}.png".format(idx)),
                        dpi=300, bbox_inches="tight")
            plt.close(fig)

        grid_inps.append(inp_img); grid_preds.append(pred_img); grid_tgts.append(tgt_img)

    # ── Save CSV ──────────────────────────────────────────────────────────────
    import csv
    csv_path = os.path.join(out_dir, "metrics_{}_test.csv".format(direction))
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader(); w.writerows(rows)

    ssims = [r["ssim"] for r in rows]
    psnrs = [r["psnr_db"] for r in rows]
    accs  = [r["pixel_acc"] for r in rows]
    print("  SSIM {:.4f}±{:.4f} | PSNR {:.2f}±{:.2f} dB | PixAcc {:.4f}±{:.4f}".format(
          np.mean(ssims), np.std(ssims),
          np.mean(psnrs), np.std(psnrs),
          np.mean(accs),  np.std(accs)))
    print("  CSV:", csv_path)

    # ── Comparison grid (first 8) ─────────────────────────────────────────────
    n_grid = min(8, len(rows))
    fig, axes = plt.subplots(n_grid, 3, figsize=(15, 5*n_grid))
    if n_grid == 1: axes = [axes]
    for r, (ai, ap, at) in enumerate(zip(grid_inps[:n_grid],
                                          grid_preds[:n_grid], grid_tgts[:n_grid])):
        for col, img in enumerate([ai, ap, at]):
            axes[r][col].imshow(img); axes[r][col].axis("off")
        axes[r][0].set_ylabel("#{} SSIM={:.4f}".format(r, rows[r]["ssim"]), fontsize=8)
    axes[0][0].set_title("Input",                fontsize=11, fontweight="bold")
    axes[0][1].set_title("GeoAgent V2 (Ours)",   fontsize=11, fontweight="bold")
    axes[0][2].set_title("Ground Truth",          fontsize=11, fontweight="bold")
    plt.suptitle("{} {} — Test Set Grid".format(prop_label, direction), fontsize=12)
    plt.tight_layout()
    grid_path = os.path.join(out_dir, "grid_{}_test.png".format(direction))
    fig.savefig(grid_path, dpi=300, bbox_inches="tight"); plt.show()
    print("  Grid:", grid_path)

    return rows, ssim_map

print("evaluate_hires() loaded.")


In [ ]:
import os, glob, torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_ckpt(tag, direction):
    """Find best (latest epoch) checkpoint in a tag dir, any epoch number."""
    d = os.path.join(CHECKPOINTS_DIR, tag)
    if not os.path.isdir(d): return None
    pattern = os.path.join(d, "v2_{}_ep*.pth".format(direction))
    ckpts = sorted(glob.glob(pattern))
    if not ckpts:                   # try alternative naming
        ckpts = sorted(glob.glob(os.path.join(d, "*.pth")))
    return ckpts[-1] if ckpts else None

# ── Evaluation configs: (tag, direction, dataset, prop_label, in_ch) ──────────
eval_configs = [
    # PorFac — 2D models (trained), 3D if available
    ("2D_PorFac_AB",  "AB", DATASET_2D_POR_FAC,  "PorFac",  3),
    ("2D_PorFac_BA",  "BA", DATASET_2D_POR_FAC,  "PorFac",  3),
    # FacPerm — 2D models (trained), 3D if available
    ("2D_FacPerm_AB", "AB", DATASET_2D_FAC_PERM, "FacPerm", 3),
    ("2D_FacPerm_BA", "BA", DATASET_2D_FAC_PERM, "FacPerm", 3),
    # 3D models (uncomment after 3D training)
    # ("3D_PorFac_AB",  "AB", DATASET_3D_POR_FAC,  "PorFac",  9),
    # ("3D_PorFac_BA",  "BA", DATASET_3D_POR_FAC,  "PorFac",  9),
    # ("3D_FacPerm_AB", "AB", DATASET_3D_FAC_PERM, "FacPerm", 9),
    # ("3D_FacPerm_BA", "BA", DATASET_3D_FAC_PERM, "FacPerm", 9),
]

for tag, direction, dataset, prop_label, in_ch in eval_configs:
    ckpt = find_ckpt(tag, direction)
    if ckpt:
        print("\n" + "=" * 65)
        print("  Evaluating: {} | {} | in_ch={} | {}".format(
              tag, direction, in_ch, os.path.basename(ckpt)))
        print("=" * 65)
        # save to v2_ prefix so comparison table tags stay consistent
        out_tag = tag.replace("2D_","v2_").replace("3D_","v2_3d_")
        evaluate_hires(ckpt, dataset, direction,
                       os.path.join(RESULTS_DIR, out_tag),
                       in_ch=in_ch, image_size=IMAGE_SIZE,
                       device=device, prop_label=prop_label)
    else:
        print("\nSkipping {} {} — checkpoint not found in {}".format(
              tag, direction, os.path.join(CHECKPOINTS_DIR, tag)))


## D. V1 vs V2 Paper Comparison Table

In [ ]:
import csv, os
import pandas as pd
import numpy as np

def get_stats(csv_path):
    if not os.path.exists(csv_path): return None
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    if not rows: return None
    ssims = [float(r["ssim"])      for r in rows]
    psnrs = [float(r["psnr_db"])   for r in rows]
    accs  = [float(r["pixel_acc"]) for r in rows]
    return {
        "SSIM":   "{:.4f}\u00b1{:.4f}".format(np.mean(ssims), np.std(ssims)),
        "PSNR":   "{:.2f}\u00b1{:.2f}".format(np.mean(psnrs), np.std(psnrs)),
        "PixAcc": "{:.4f}\u00b1{:.4f}".format(np.mean(accs),  np.std(accs)),
        "N":      len(rows),
        "ssim_mean": np.mean(ssims),
    }

# V1 results saved as: RESULTS_DIR/PorFac/metrics_AB_test.csv
# V2 results saved as: RESULTS_DIR/v2_PorFac_AB/metrics_AB_test.csv
configs = [
    ("V1 (ngf=64, 2D)",   "PorFac",      "AB", "Porosity\u2192Facies"),
    ("V1 (ngf=64, 2D)",   "PorFac",      "BA", "Facies\u2192Porosity"),
    ("V1 (ngf=64, 2D)",   "FacPerm_AB",  "AB", "Facies\u2192Permeability"),
    ("V1 (ngf=64, 2D)",   "FacPerm_BA",  "BA", "Permeability\u2192Facies"),
    ("V2 (ngf=128, 2D)",  "v2_PorFac_AB","AB", "Porosity\u2192Facies"),
    ("V2 (ngf=128, 2D)",  "v2_PorFac_BA","BA", "Facies\u2192Porosity"),
    ("V2 (ngf=128, 2D)",  "v2_FacPerm_AB","AB","Facies\u2192Permeability"),
    ("V2 (ngf=128, 2D)",  "v2_FacPerm_BA","BA","Permeability\u2192Facies"),
]

rows = []
for model_ver, tag, direction, label in configs:
    csv_path = os.path.join(RESULTS_DIR, tag,
                            "metrics_" + direction + "_test.csv")
    stats = get_stats(csv_path)
    if stats is None:
        print("Not found:", csv_path)
        continue
    rows.append({
        "Model":     model_ver,
        "Direction": label,
        "SSIM":      stats["SSIM"],
        "PSNR(dB)":  stats["PSNR"],
        "PixAcc":    stats["PixAcc"],
        "N":         stats["N"],
    })

if rows:
    df = pd.DataFrame(rows)
    print("=== V1 vs V2 — All 4 Directions ===\n")
    print(df.to_string(index=False))
    out = os.path.join(RESULTS_DIR, "comparison_v1_v2_all.csv")
    df.to_csv(out, index=False)
    print("\nSaved:", out)
else:
    print("No results yet. Run Cell C (evaluate_hires) first.")


# GeoAgent-LLM v3 — Complete Notebook
**Author:** Abdulrahman Al-Fakih  
Novel contributions: LLM-guided GAN refinement · 3D context · ablation · advanced validation  
Target: High-impact journal + patent filing

In [ ]:
# ── v3 Dependencies ──────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "anthropic", "lpips", "scipy", "scikit-image",
                "--quiet", "--break-system-packages"], check=False)

import torch
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU    :", torch.cuda.get_device_name(0))
print()
# All paths/config already defined in Cell 15 (master config)
print("Paths from master config:")
print("  BASE_DIR       :", BASE_DIR)
print("  CHECKPOINTS_DIR:", CHECKPOINTS_DIR)
print("  RESULTS_DIR    :", RESULTS_DIR)
print("  DATASET_POR_FAC :", DATASET_POR_FAC)
print("  DATASET_FAC_PERM:", DATASET_FAC_PERM)
print("  NGF={} | IMAGE_SIZE={} | MC_SAMPLES={}".format(NGF, IMAGE_SIZE, MC_SAMPLES))
print("  LLM_ENABLED    :", LLM_ENABLED)


## 1. Dataset Loaders — 2D and 3D
Both loaders use the **same paired PNG format** (left=A, right=B). The 3D loader additionally indexes (i,j,k) coordinates from filenames.

In [ ]:
%%writefile geoagent_dataset_2d.py
"""2D Paired Dataset loader with zip support. Author: Abdulrahman Al-Fakih"""
import os, re, zipfile, random
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

class PairedGeoDataset2D(Dataset):
    def __init__(self, root, direction="AB", image_size=256, split="train",
                 train_ratio=0.70, val_ratio=0.15, seed=42):
        self.direction  = direction
        self.image_size = image_size
        self.root = self._maybe_extract(root)
        files = sorted([
            os.path.join(dp, f)
            for dp, _, fns in os.walk(self.root)
            for f in fns if os.path.splitext(f)[1].lower() in EXTS
        ])
        if not files:
            raise ValueError("No images found in: " + self.root)
        rng = random.Random(seed)
        rng.shuffle(files)
        n = len(files)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)
        splits = {"train": files[:n_train],
                  "val":   files[n_train:n_train+n_val],
                  "test":  files[n_train+n_val:]}
        self.files = splits[split]
        self.resize  = transforms.Resize((image_size, image_size), antialias=True)
        self.to_tensor = transforms.ToTensor()
        print("[Dataset2D] {} slices | direction={} | split={}".format(
              len(self.files), direction, split))
    @staticmethod
    def _maybe_extract(root):
        if os.path.isdir(root): return root
        zips = [f for f in [root, root+".zip"] if os.path.isfile(f)]
        if not zips: raise FileNotFoundError("Not found: " + root)
        out = zips[0].replace(".zip", "")
        with zipfile.ZipFile(zips[0]) as z: z.extractall(os.path.dirname(out))
        print("Extracted:", zips[0], "->", out)
        return out
    def _load_half(self, path, side):
        img = Image.open(path).convert("RGB")
        w, h = img.size
        mid = w // 2
        box = (0, 0, mid, h) if side == "A" else (mid, 0, w, h)
        crop = self.resize(img.crop(box))
        t = self.to_tensor(crop)       # [0,1]
        return t * 2.0 - 1.0          # [-1,1]
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        path = self.files[idx]
        side_in  = "A" if self.direction == "AB" else "B"
        side_out = "B" if self.direction == "AB" else "A"
        inp = self._load_half(path, side_in)
        tgt = self._load_half(path, side_out)
        return {"input": inp, "target": tgt, "path": path}

def build_dataloaders_2d(root, direction="AB", image_size=256,
                         batch_size=8, num_workers=0):
    loaders = {}
    for split in ["train","val","test"]:
        ds = PairedGeoDataset2D(root, direction, image_size, split)
        loaders[split] = DataLoader(
            ds, batch_size=batch_size if split=="train" else 1,
            shuffle=(split=="train"), num_workers=num_workers,
            pin_memory=torch.cuda.is_available())
    return loaders


## 2. LLM Semantic Reasoning Interface
Uses **Claude Vision API** to assess generated reservoir images and return
structured JSON: failure mode, geological assessment, recommended action.
Falls back gracefully to rule-based if API key not set.

In [ ]:
%%writefile geoagent_llm.py
"""LLM Semantic Reasoning for GeoAgent. Author: Abdulrahman Al-Fakih"""
import base64, io, json, os
import numpy as np
from PIL import Image
import torch

def _tensor_to_png_b64(t):
    """Convert (1,3,H,W) or (3,H,W) tensor [-1,1] to base64 PNG string."""
    if t.dim() == 4: t = t.squeeze(0)
    if t.shape[0] > 3: t = t[3:6]  # center channels of 9-ch 3D input
    arr = ((t.clamp(-1,1)+1)/2 * 255).permute(1,2,0).cpu().numpy().astype("uint8")
    img = Image.fromarray(arr)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

SYSTEM_PROMPT = (
    "You are a geoscience expert specializing in reservoir characterization and "
    "generative AI for subsurface property prediction. You assess AI-generated "
    "geological property maps (facies, porosity, permeability) against ground truth."
)

USER_TEMPLATE = (
    "Assess this GAN-generated reservoir property image. "
    "Left panel: input property. Center: GeoAgent prediction. Right: ground truth. "
    "Direction: {direction}. SSIM score: {ssim:.4f}. Attempt: {attempt}/{max_attempts}. "
    "Previous failure mode: {prev_failure}.\n\n"
    "Return ONLY a JSON object with these exact keys:\n"
    "  quality_score: float 0-1 (your geological quality assessment)\n"
    "  failure_mode: one of none|texture_blur|boundary_shift|color_drift|structural_loss|mixed\n"
    "  geological_assessment: brief text (max 2 sentences)\n"
    "  recommended_action: one of accept|flip_horizontal|flip_vertical|rotate_90|contrast_enhance|noise_inject\n"
    "  accept: true or false\n"
    "  confidence: float 0-1\n"
    "  spatial_issues: list of strings describing problem areas (empty if none)"
)

FALLBACK_RULES = {
    (0.90, 1.01): {"quality_score":0.95,"failure_mode":"none","geological_assessment":"High quality output.","recommended_action":"accept","accept":True,"confidence":0.95,"spatial_issues":[]},
    (0.80, 0.90): {"quality_score":0.82,"failure_mode":"texture_blur","geological_assessment":"Minor blurring detected.","recommended_action":"contrast_enhance","accept":False,"confidence":0.80,"spatial_issues":["fine texture"]},
    (0.70, 0.80): {"quality_score":0.72,"failure_mode":"boundary_shift","geological_assessment":"Boundary displacement observed.","recommended_action":"flip_horizontal","accept":False,"confidence":0.70,"spatial_issues":["boundary regions"]},
    (0.00, 0.70): {"quality_score":0.55,"failure_mode":"structural_loss","geological_assessment":"Major structural mismatch.","recommended_action":"rotate_90","accept":False,"confidence":0.60,"spatial_issues":["global structure"]},
}

def _fallback(ssim):
    for (lo, hi), result in FALLBACK_RULES.items():
        if lo <= ssim < hi: return dict(result)
    return dict(list(FALLBACK_RULES.values())[-1])

def llm_assess(inp_t, pred_t, tgt_t, ssim, direction="AB",
               attempt=1, max_attempts=3, prev_failure="none",
               api_key="", model="claude-sonnet-5"):
    """Query Claude Vision API. Returns dict with quality_score, failure_mode, etc."""
    if not api_key:
        result = _fallback(ssim)
        result["source"] = "rule_based"
        return result
    try:
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
        # Build 3-panel comparison image
        def t2arr(t):
            if t.dim()==4: t=t.squeeze(0)
            if t.shape[0]>3: t=t[3:6]
            return ((t.clamp(-1,1)+1)/2*255).permute(1,2,0).cpu().numpy().astype("uint8")
        panels = [t2arr(inp_t), t2arr(pred_t), t2arr(tgt_t)]
        h, w = panels[0].shape[:2]
        combo = np.zeros((h, w*3+4, 3), dtype="uint8")
        for i, p in enumerate(panels): combo[:, i*(w+2):i*(w+2)+w] = p
        buf = io.BytesIO()
        Image.fromarray(combo).save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        prompt = USER_TEMPLATE.format(
            direction=direction, ssim=ssim,
            attempt=attempt, max_attempts=max_attempts,
            prev_failure=prev_failure)
        msg = client.messages.create(
            model=model, max_tokens=512,
            system=SYSTEM_PROMPT,
            messages=[{"role":"user","content":[
                {"type":"image","source":{"type":"base64","media_type":"image/png","data":b64}},
                {"type":"text","text":prompt}
            ]}])
        text = msg.content[0].text.strip()
        # Extract JSON from response
        start = text.find("{")
        end   = text.rfind("}")+1
        result = json.loads(text[start:end])
        result["source"] = "claude_api"
        return result
    except Exception as e:
        result = _fallback(ssim)
        result["source"] = "fallback_error:" + str(e)[:60]
        return result


## 3. GeoAgent-LLM — ReAct Loop with LLM OBSERVE Step
REASON → ACT → **LLM OBSERVE** → ITERATE.  
The LLM diagnoses failure mode and prescribes targeted augmentation (not random TTA).

In [ ]:
%%writefile geoagent_agent_llm.py
"""GeoAgent-LLM: ReAct loop with LLM semantic reasoning. Author: Abdulrahman Al-Fakih"""
import time, json
import torch
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field
from typing import List, Optional

# ── Metrics ──────────────────────────────────────────────────────────────────
def compute_ssim(pred, target, window_size=11):
    p = (pred.clamp(-1,1)+1)/2
    t = (target.clamp(-1,1)+1)/2
    if p.shape[1] > 3: p = p[:,3:6]; t = t[:,3:6]
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(p,window_size,1,window_size//2)
    mu_t = F.avg_pool2d(t,window_size,1,window_size//2)
    sp = F.avg_pool2d(p*p,window_size,1,window_size//2)-mu_p**2
    st = F.avg_pool2d(t*t,window_size,1,window_size//2)-mu_t**2
    spt= F.avg_pool2d(p*t,window_size,1,window_size//2)-mu_p*mu_t
    num=(2*mu_p*mu_t+C1)*(2*spt+C2)
    den=(mu_p**2+mu_t**2+C1)*(sp+st+C2)
    return float((num/den).mean())

def compute_psnr(pred, target):
    p=(pred.clamp(-1,1)+1)/2; t=(target.clamp(-1,1)+1)/2
    if p.shape[1]>3: p=p[:,3:6]; t=t[:,3:6]
    mse=((p-t)**2).mean().item()
    return 100.0 if mse<1e-10 else 10*float(np.log10(1.0/mse))

def compute_pixel_accuracy(pred, target, thr=0.05):
    p=(pred.clamp(-1,1)+1)/2; t=(target.clamp(-1,1)+1)/2
    if p.shape[1]>3: p=p[:,3:6]; t=t[:,3:6]
    return float((torch.abs(p-t)<thr).float().mean())

# ── LLM-guided TTA actions ───────────────────────────────────────────────────
def apply_action(x, action):
    """Apply LLM-recommended augmentation to input tensor."""
    if action == "flip_horizontal":   return torch.flip(x, [-1])
    if action == "flip_vertical":     return torch.flip(x, [-2])
    if action == "rotate_90":         return torch.rot90(x, 1, [-2,-1])
    if action == "contrast_enhance":  return (x * 1.2).clamp(-1, 1)
    if action == "noise_inject":      return (x + torch.randn_like(x)*0.05).clamp(-1,1)
    return x  # accept / unknown

def undo_action(pred, action):
    """Invert the augmentation applied to recover aligned prediction."""
    if action == "flip_horizontal":   return torch.flip(pred, [-1])
    if action == "flip_vertical":     return torch.flip(pred, [-2])
    if action == "rotate_90":         return torch.rot90(pred, -1, [-2,-1])
    return pred

# ── Result dataclass ─────────────────────────────────────────────────────────
@dataclass
class AgentResult:
    predicted_image:    torch.Tensor
    ssim_score:         float
    psnr_score:         float
    pixel_accuracy:     float
    llm_quality_score:  float
    n_iterations:       int
    accepted:           bool
    failure_mode:       str
    llm_source:         str
    geological_assessment: str
    reasoning_trace:    List[dict] = field(default_factory=list)
    elapsed_ms:         float = 0.0

# ── GeoAgent-LLM ─────────────────────────────────────────────────────────────
class GeoAgentLLM:
    def __init__(self, model, device, ssim_threshold=0.70,
                 max_iterations=3, api_key="",
                 llm_model="claude-sonnet-5", direction="AB"):
        self.model          = model
        self.device         = device
        self.ssim_threshold = ssim_threshold
        self.max_iterations = max_iterations
        self.api_key        = api_key
        self.llm_model      = llm_model
        self.direction      = direction
    @torch.no_grad()
    def _infer(self, x):
        self.model.G.eval()
        return self.model.G(x.to(self.device)).cpu()
    def run(self, inp, target=None):
        from geoagent_llm import llm_assess
        t0 = time.time()
        trace = []
        best_pred  = None
        best_ssim  = -1.0
        prev_failure = "none"
        action = "accept"
        for iteration in range(1, self.max_iterations+1):
            # ── REASON ──
            x_aug = apply_action(inp, action) if iteration > 1 else inp
            # ── ACT ──
            pred_aug = self._infer(x_aug)
            pred = undo_action(pred_aug, action)
            ssim = compute_ssim(pred, target) if target is not None else 0.75
            psnr = compute_psnr(pred, target) if target is not None else 0.0
            acc  = compute_pixel_accuracy(pred, target) if target is not None else 0.0
            if ssim > best_ssim:
                best_ssim = ssim; best_pred = pred.clone()
            # ── LLM OBSERVE ──
            tgt_for_llm = target if target is not None else pred
            llm_out = llm_assess(
                inp, pred, tgt_for_llm, ssim,
                direction=self.direction,
                attempt=iteration, max_attempts=self.max_iterations,
                prev_failure=prev_failure,
                api_key=self.api_key, model=self.llm_model)
            trace.append({
                "iteration": iteration,
                "ssim": round(ssim, 4),
                "psnr": round(psnr, 2),
                "pixel_acc": round(acc, 4),
                "action_applied": action,
                "llm_failure_mode": llm_out.get("failure_mode","unknown"),
                "llm_quality_score": llm_out.get("quality_score", ssim),
                "llm_accept": llm_out.get("accept", ssim>=self.ssim_threshold),
                "llm_source": llm_out.get("source","rule_based"),
                "geological_assessment": llm_out.get("geological_assessment",""),
            })
            # ── ITERATE or ACCEPT ──
            accepted = llm_out.get("accept", ssim >= self.ssim_threshold)
            if accepted or iteration == self.max_iterations:
                break
            action = llm_out.get("recommended_action", "flip_horizontal")
            prev_failure = llm_out.get("failure_mode", "unknown")
        elapsed = (time.time()-t0)*1000
        last = trace[-1]
        return AgentResult(
            predicted_image    = best_pred,
            ssim_score         = best_ssim,
            psnr_score         = last["psnr"],
            pixel_accuracy     = last["pixel_acc"],
            llm_quality_score  = last["llm_quality_score"],
            n_iterations       = len(trace),
            accepted           = last["llm_accept"],
            failure_mode       = last["llm_failure_mode"],
            llm_source         = last["llm_source"],
            geological_assessment = last["geological_assessment"],
            reasoning_trace    = trace,
            elapsed_ms         = elapsed,
        )


## 4. Uncertainty Quantification — Monte Carlo Dropout
Run N forward passes with dropout **active** at test time.  
Pixel-wise `std` = aleatoric + epistemic uncertainty map (novel for geoscience GANs).

In [ ]:
%%writefile geoagent_uncertainty.py
"""Monte Carlo Dropout uncertainty for GeoAgent. Author: Abdulrahman Al-Fakih"""
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

def mc_dropout_predict(model, x, n_samples=20, device=None):
    """Run N forward passes with dropout active. Returns mean pred + uncertainty map."""
    device = device or next(model.G.parameters()).device
    x = x.to(device)
    model.G.train()  # activates dropout at test time
    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            p = model.G(x)
            if p.shape[1] > 3: p = p[:,3:6]
            preds.append(p.cpu())
    model.G.eval()
    stack = torch.stack(preds, dim=0)          # (N, B, 3, H, W)
    mean  = stack.mean(0)                       # (B, 3, H, W)
    std   = stack.std(0)                        # (B, 3, H, W) — uncertainty
    # Coefficient of variation per pixel
    eps   = 1e-8
    cv    = std / (torch.abs(mean) + eps)      # (B, 3, H, W)
    return mean, std, cv

def save_uncertainty_panel(inp_t, pred_mean, pred_std, tgt_t, out_path, title=""):
    """Save 4-panel: Input | Pred Mean | Uncertainty | Ground Truth at 300 DPI."""
    def t2img(t):
        if t.dim()==4: t=t.squeeze(0)
        if t.shape[0]>3: t=t[3:6]
        return ((t.clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    inp_arr  = t2img(inp_t)
    mean_arr = t2img(pred_mean)
    std_arr  = pred_std.squeeze(0).mean(0).numpy()   # scalar uncertainty map
    tgt_arr  = t2img(tgt_t)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(inp_arr);  axes[0].set_title("Input", fontsize=11); axes[0].axis("off")
    axes[1].imshow(mean_arr); axes[1].set_title("GeoAgent-LLM Mean", fontsize=11); axes[1].axis("off")
    im = axes[2].imshow(std_arr, cmap="hot", vmin=0)
    axes[2].set_title("Epistemic Uncertainty", fontsize=11); axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], shrink=0.8, label="Std Dev")
    axes[3].imshow(tgt_arr);  axes[3].set_title("Ground Truth", fontsize=11); axes[3].axis("off")
    if title: fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    parent = os.path.dirname(os.path.abspath(out_path))
    os.makedirs(parent, exist_ok=True)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return std_arr.mean()


## 5. Advanced Evaluation Suite
Metrics for publication + patent validation:
- **Variogram** (spatial continuity, horizontal + vertical)
- **KL Divergence** (histogram distribution match)
- **Earth Mover's Distance** (Wasserstein, property value distribution)
- **Spatial SSIM map** (per-pixel quality heatmap)
- **FID** (Frechet Inception Distance, GAN quality standard)
- **LPIPS** (perceptual similarity)
- **MC Uncertainty** mean/max over test set

In [ ]:
%%writefile geoagent_evaluate_advanced.py
"""Advanced evaluation suite for GeoAgent-LLM. Author: Abdulrahman Al-Fakih"""
import os, json, csv
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import wasserstein_distance, entropy
from scipy.ndimage import uniform_filter

# ── Variogram ────────────────────────────────────────────────────────────────
def compute_variogram(arr2d, max_lag=50, n_lags=25, direction="horizontal"):
    """Experimental variogram: gamma(h)=1/(2N)*sum[(z_i - z_{i+h})^2]."""
    lags = np.linspace(1, max_lag, n_lags, dtype=int)
    gamma = []
    for h in lags:
        if direction == "horizontal":
            diff = arr2d[:, h:] - arr2d[:, :-h]
        else:  # vertical
            diff = arr2d[h:, :] - arr2d[:-h, :]
        gamma.append(float(0.5 * np.mean(diff**2)))
    return np.array(lags), np.array(gamma)

def variogram_mae(gamma_real, gamma_gen):
    return float(np.mean(np.abs(gamma_real - gamma_gen)))

# ── KL Divergence ────────────────────────────────────────────────────────────
def compute_kl_divergence(pred_arr, tgt_arr, bins=64):
    p_hist, edges = np.histogram(pred_arr.ravel(), bins=bins, range=(0,1), density=True)
    t_hist, _    = np.histogram(tgt_arr.ravel(),  bins=bins, range=(0,1), density=True)
    eps = 1e-10
    p_hist = p_hist + eps; t_hist = t_hist + eps
    p_hist /= p_hist.sum(); t_hist /= t_hist.sum()
    return float(entropy(p_hist, t_hist))

# ── Earth Mover Distance ─────────────────────────────────────────────────────
def compute_emd(pred_arr, tgt_arr):
    return float(wasserstein_distance(pred_arr.ravel(), tgt_arr.ravel()))

# ── Spatial SSIM map ─────────────────────────────────────────────────────────
def spatial_ssim_map(pred_arr, tgt_arr, win=11):
    """Per-pixel SSIM map for quality heatmap visualization."""
    C1, C2 = 0.01**2, 0.03**2
    mu_p = uniform_filter(pred_arr.astype(float), win)
    mu_t = uniform_filter(tgt_arr.astype(float), win)
    sp  = uniform_filter(pred_arr**2, win) - mu_p**2
    st  = uniform_filter(tgt_arr**2, win) - mu_t**2
    spt = uniform_filter(pred_arr*tgt_arr, win) - mu_p*mu_t
    num = (2*mu_p*mu_t+C1)*(2*spt+C2)
    den = (mu_p**2+mu_t**2+C1)*(sp+st+C2)
    return num / (den + 1e-10)

# ── FID (simplified, using activation statistics) ─────────────────────────────
def compute_fid_features(tensor_list, device):
    """Extract VGG features for FID computation."""
    from torchvision import models
    vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].to(device).eval()
    feats = []
    with torch.no_grad():
        for t in tensor_list:
            if t.dim()==3: t=t.unsqueeze(0)
            if t.shape[1]>3: t=t[:,3:6]
            p = (t.clamp(-1,1)+1)/2
            f = vgg(p.to(device)).mean([2,3]).cpu().numpy()
            feats.append(f[0])
    return np.array(feats)

def compute_fid(real_feats, gen_feats):
    from scipy.linalg import sqrtm
    mu_r, mu_g = real_feats.mean(0), gen_feats.mean(0)
    cov_r = np.cov(real_feats, rowvar=False)
    cov_g = np.cov(gen_feats,  rowvar=False)
    diff  = mu_r - mu_g
    cov_mean = sqrtm(cov_r @ cov_g)
    if np.iscomplexobj(cov_mean): cov_mean = cov_mean.real
    return float(diff @ diff + np.trace(cov_r + cov_g - 2*cov_mean))

# ── LPIPS ────────────────────────────────────────────────────────────────────
def try_lpips(pred_t, tgt_t, device):
    try:
        import lpips
        fn = lpips.LPIPS(net="vgg").to(device)
        p = pred_t.clamp(-1,1)
        t = tgt_t.clamp(-1,1)
        if p.shape[1]>3: p=p[:,3:6]; t=t[:,3:6]
        with torch.no_grad():
            return float(fn(p.to(device), t.to(device)).mean())
    except Exception:
        return None

# ── Main evaluation function ──────────────────────────────────────────────────
def evaluate_full(model, loader, agent, direction, out_dir, device,
                  mode="2D", n_mc=20, api_key="", llm_model="claude-sonnet-5"):
    """
    Full evaluation: standard metrics + variogram + KL + EMD + SSIM map + FID + uncertainty.
    Returns list of per-sample result dicts.
    """
    from geoagent_uncertainty import mc_dropout_predict, save_uncertainty_panel
    os.makedirs(out_dir, exist_ok=True)
    indiv_dir = os.path.join(out_dir, "individual")
    uncert_dir= os.path.join(out_dir, "uncertainty")
    vario_dir = os.path.join(out_dir, "variograms")
    ssim_dir  = os.path.join(out_dir, "ssim_maps")
    for d in [indiv_dir, uncert_dir, vario_dir, ssim_dir]:
        os.makedirs(d, exist_ok=True)

    rows = []
    real_feats, gen_feats = [], []
    variogram_real_h, variogram_gen_h = [], []
    variogram_real_v, variogram_gen_v = [], []
    uncertainty_vals = []

    print("Evaluating {} test samples [{}] ...".format(len(loader), mode))
    for idx, batch in enumerate(loader):
        inp  = batch["input"]
        tgt  = batch["target"]
        path = batch["path"][0]

        # ── GeoAgent-LLM inference ──
        result = agent.run(inp, tgt)
        pred   = result.predicted_image

        # ── Monte Carlo uncertainty ──
        pred_mean, pred_std, pred_cv = mc_dropout_predict(model, inp, n_mc, device)
        unc_mean = float(pred_std.mean())
        unc_max  = float(pred_std.max())
        uncertainty_vals.append(unc_mean)

        # ── Numpy arrays for spatial metrics ──
        def t2gray(t):
            if t.dim()==4: t=t.squeeze(0)
            if t.shape[0]>3: t=t[3:6]
            return ((t.clamp(-1,1)+1)/2).mean(0).numpy()
        pred_arr = t2gray(pred)
        tgt_arr  = t2gray(tgt)

        # ── Spatial metrics ──
        kl  = compute_kl_divergence(pred_arr, tgt_arr)
        emd = compute_emd(pred_arr, tgt_arr)
        ssim_map = spatial_ssim_map(pred_arr, tgt_arr)
        ssim_mean = float(ssim_map.mean())

        # ── Variogram (horizontal + vertical) ──
        lag, vr_h = compute_variogram(tgt_arr,  direction="horizontal")
        lag, vg_h = compute_variogram(pred_arr, direction="horizontal")
        lag, vr_v = compute_variogram(tgt_arr,  direction="vertical")
        lag, vg_v = compute_variogram(pred_arr, direction="vertical")
        variogram_real_h.append(vr_h); variogram_gen_h.append(vg_h)
        variogram_real_v.append(vr_v); variogram_gen_v.append(vg_v)
        vario_mae_h = variogram_mae(vr_h, vg_h)
        vario_mae_v = variogram_mae(vr_v, vg_v)

        # ── LPIPS ──
        lpips_val = try_lpips(pred.unsqueeze(0) if pred.dim()==3 else pred, tgt, device)

        # ── FID feature accumulation ──
        real_feats.append(tgt.squeeze(0) if tgt.dim()==4 else tgt)
        gen_feats.append(pred.squeeze(0) if pred.dim()==4 else pred)

        # ── Save uncertainty panel ──
        fname = os.path.splitext(os.path.basename(path))[0]
        unc_path = os.path.join(uncert_dir, mode+"_"+direction+"_"+fname+".png")
        save_uncertainty_panel(inp, pred_mean, pred_std, tgt, unc_path,
                               title=mode+" | "+direction+" | SSIM={:.4f}".format(result.ssim_score))

        # ── Save SSIM map ──
        smap_path = os.path.join(ssim_dir, mode+"_"+direction+"_"+fname+".png")
        fig_s, ax_s = plt.subplots(1,1,figsize=(5,5))
        im_s = ax_s.imshow(ssim_map, cmap="RdYlGn", vmin=0, vmax=1)
        plt.colorbar(im_s, ax=ax_s, shrink=0.8, label="SSIM")
        ax_s.set_title("Spatial SSIM Map", fontsize=10); ax_s.axis("off")
        fig_s.savefig(smap_path, dpi=200, bbox_inches="tight")
        plt.close(fig_s)

        row = {
            "mode": mode, "direction": direction, "filename": fname,
            "ssim":         round(result.ssim_score, 4),
            "psnr_db":      round(result.psnr_score, 2),
            "pixel_acc":    round(result.pixel_accuracy, 4),
            "ssim_spatial": round(ssim_mean, 4),
            "lpips":        round(lpips_val, 4) if lpips_val else None,
            "kl_divergence":round(kl, 6),
            "emd":          round(emd, 6),
            "vario_mae_h":  round(vario_mae_h, 6),
            "vario_mae_v":  round(vario_mae_v, 6),
            "unc_mean":     round(unc_mean, 6),
            "unc_max":      round(unc_max, 6),
            "llm_quality":  round(result.llm_quality_score, 4),
            "llm_failure":  result.failure_mode,
            "llm_accept":   result.accepted,
            "llm_source":   result.llm_source,
            "n_iters":      result.n_iterations,
            "elapsed_ms":   round(result.elapsed_ms, 1),
        }
        rows.append(row)

        if (idx+1) % 20 == 0 or (idx+1) == len(loader):
            ms = round(np.mean([r["ssim"] for r in rows]), 4)
            print("  [{}/{}] SSIM={:.4f} KL={:.4f} EMD={:.4f} Unc={:.4f}".format(
                  idx+1, len(loader), ms,
                  np.mean([r["kl_divergence"] for r in rows]),
                  np.mean([r["emd"] for r in rows]),
                  np.mean([r["unc_mean"] for r in rows])))

    # ── FID ──────────────────────────────────────────────────────────────────
    try:
        rf = compute_fid_features(real_feats, device)
        gf = compute_fid_features(gen_feats,  device)
        fid_val = compute_fid(rf, gf)
    except Exception as e:
        fid_val = -1.0
        print("FID error:", e)

    # ── Aggregate variogram plot ──────────────────────────────────────────────
    mean_vr_h = np.mean(variogram_real_h, 0)
    mean_vg_h = np.mean(variogram_gen_h,  0)
    mean_vr_v = np.mean(variogram_real_v, 0)
    mean_vg_v = np.mean(variogram_gen_v,  0)
    lag_arr = np.linspace(1, 50, 25)
    fig_v, axes_v = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (vr, vg, title) in zip(axes_v, [
            (mean_vr_h, mean_vg_h, "Horizontal Variogram"),
            (mean_vr_v, mean_vg_v, "Vertical Variogram")]):
        ax.plot(lag_arr, vr, "b-o", markersize=4, label="Ground Truth")
        ax.plot(lag_arr, vg, "r-s", markersize=4, label="GeoAgent-LLM")
        ax.fill_between(lag_arr, vr, vg, alpha=0.15, color="gray")
        ax.set_xlabel("Lag distance (pixels)", fontsize=10)
        ax.set_ylabel("Semivariance", fontsize=10)
        ax.set_title(title+" | "+mode+" | "+direction, fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    vario_out = os.path.join(vario_dir, "variogram_{}_{}.png".format(mode, direction))
    fig_v.savefig(vario_out, dpi=300, bbox_inches="tight")
    plt.close(fig_v)
    print("Variogram saved:", vario_out)

    # ── Summary JSON ─────────────────────────────────────────────────────────
    summary = {
        "mode": mode, "direction": direction, "n_samples": len(rows),
        "ssim_mean":    round(float(np.mean([r["ssim"]         for r in rows])), 4),
        "ssim_std":     round(float(np.std( [r["ssim"]         for r in rows])), 4),
        "psnr_mean_db": round(float(np.mean([r["psnr_db"]      for r in rows])), 2),
        "psnr_std_db":  round(float(np.std( [r["psnr_db"]      for r in rows])), 2),
        "pixel_acc":    round(float(np.mean([r["pixel_acc"]    for r in rows])), 4),
        "kl_div":       round(float(np.mean([r["kl_divergence"]for r in rows])), 6),
        "emd":          round(float(np.mean([r["emd"]          for r in rows])), 6),
        "vario_mae_h":  round(float(np.mean([r["vario_mae_h"]  for r in rows])), 6),
        "vario_mae_v":  round(float(np.mean([r["vario_mae_v"]  for r in rows])), 6),
        "lpips_mean":   round(float(np.mean([r["lpips"] for r in rows if r["lpips"] is not None])),4) if any(r["lpips"] for r in rows) else None,
        "fid":          round(fid_val, 2),
        "unc_mean":     round(float(np.mean([r["unc_mean"]     for r in rows])), 6),
        "unc_max":      round(float(np.max( [r["unc_max"]      for r in rows])), 6),
        "llm_accept_rate": round(float(np.mean([r["llm_accept"] for r in rows])), 4),
        "llm_quality_mean":round(float(np.mean([r["llm_quality"]for r in rows])), 4),
        "mean_iters":   round(float(np.mean([r["n_iters"]      for r in rows])), 2),
    }
    print("=== {} {} Summary ===".format(mode, direction))
    for k,v in summary.items(): print("  {}: {}".format(k, v))

    # ── Save CSV + JSON ───────────────────────────────────────────────────────
    csv_path = os.path.join(out_dir, "metrics_{}_{}.csv".format(mode, direction))
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader(); w.writerows(rows)
    json_path = os.path.join(out_dir, "summary_{}_{}.json".format(mode, direction))
    with open(json_path, "w") as f: json.dump(summary, f, indent=2)
    print("Saved:", csv_path)
    print("Saved:", json_path)
    return rows, summary


In [ ]:
import inspect
from geoagent_model import Pix2PixModel
print(inspect.signature(Pix2PixModel.__init__))

## 6. Training — 2D and 3D, All 4 Directions
Runs 8 training jobs total (2 modes × 4 directions).  
Each saves checkpoints + loss log. Run one block at a time (~2h each on RTX 4500 Ada).

In [ ]:
import subprocess, sys, os

def run_training(data_root, direction, tag, mode="3D", epochs=300, batch=8):
    """Run geoagent_train_v2.py. mode: '3D' (in_ch=9) or '2D' (in_ch=3)."""
    ckpt_dir = os.path.join(CHECKPOINTS_DIR, tag)
    os.makedirs(ckpt_dir, exist_ok=True)
    in_ch = 9 if mode == "3D" else 3
    cmd = [
        sys.executable, "geoagent_train_v2.py",
        "--data_root",  data_root,
        "--direction",  direction,
        "--in_ch",      str(in_ch),       # ← required
        "--epochs",     str(epochs),
        "--batch_size", str(batch),
        "--image_size", str(IMAGE_SIZE),
        "--save_dir",   ckpt_dir,
        "--num_workers","0",
        "--ngf",        str(NGF),
    ]
    print("=" * 65)
    print("  Training | mode:{} | dir:{} | tag:{} | in_ch:{}".format(
          mode, direction, tag, in_ch))
    print("=" * 65)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout: print(line, end="")
    proc.wait()
    status = "DONE" if proc.returncode == 0 else "ERROR({})".format(proc.returncode)
    print(status, "->", tag)
    return ckpt_dir

# ── Uncomment one block at a time and run ─────────────────────────────────────
# PorFac 3D
# ckpt_3d_PF_AB = run_training(DATASET_3D_POR_FAC,  "AB", "3D_PorFac_AB",  mode="3D")
# ckpt_3d_PF_BA = run_training(DATASET_3D_POR_FAC,  "BA", "3D_PorFac_BA",  mode="3D")
# FacPerm 3D
# ckpt_3d_FP_AB = run_training(DATASET_3D_FAC_PERM, "AB", "3D_FacPerm_AB", mode="3D")
# ckpt_3d_FP_BA = run_training(DATASET_3D_FAC_PERM, "BA", "3D_FacPerm_BA", mode="3D")
# PorFac 2D (ablation)
# ckpt_2d_PF_AB = run_training(DATASET_2D_POR_FAC,  "AB", "2D_PorFac_AB",  mode="2D")
# ckpt_2d_PF_BA = run_training(DATASET_2D_POR_FAC,  "BA", "2D_PorFac_BA",  mode="2D")
# FacPerm 2D (ablation)
# ckpt_2d_FP_AB = run_training(DATASET_2D_FAC_PERM, "AB", "2D_FacPerm_AB", mode="2D")
# ckpt_2d_FP_BA = run_training(DATASET_2D_FAC_PERM, "BA", "2D_FacPerm_BA", mode="2D")

print("run_training() ready. Uncomment one block at a time (~2h each on RTX 4500 Ada).")


In [ ]:
train_path = r"D:\First year\Papers\GeoAgent\geoagent_train.py"
content = open(train_path, encoding="utf-8").read()

if 'add_argument("--use_3d"' not in content:
    # Find the parse_args line and insert before it (handles any indent/format)
    lines = content.split('\n')
    for i, line in enumerate(lines):
        if 'parse_args' in line:
            indent = ' ' * (len(line) - len(line.lstrip()))
            lines.insert(i, indent + 'parser.add_argument("--use_3d", action="store_true", default=False)')
            print(f"Inserted at line {i}: {lines[i].strip()}")
            break
    content = '\n'.join(lines)
    open(train_path, "w", encoding="utf-8").write(content)
else:
    print("Already present")

# Verify all 3 lines
print("\nFinal check:")
for l in content.split('\n'):
    if 'use_3d' in l or '3d_dataloaders' in l:
        print(" ", l.strip())

In [ ]:
ckpt_3d_PF_AB = run_training(DATASET_3D_POR_FAC, "AB", "3D_PorFac_AB", mode="3D")

In [ ]:
# Run now (PorFac BA):
ckpt_3d_PF_BA = run_training(DATASET_3D_POR_FAC, "BA", "3D_PorFac_BA", mode="3D")

In [ ]:
# FacPerm AB:
ckpt_3d_FP_AB = run_training(DATASET_3D_FAC_PERM, "AB", "3D_FacPerm_AB", mode="3D")

In [ ]:
# FacPerm BA:
ckpt_3d_FP_BA = run_training(DATASET_3D_FAC_PERM, "BA", "3D_FacPerm_BA", mode="3D")

In [ ]:
import os
root = DATASET_3D_POR_FAC
print("Root:", root)
print("Contents:", os.listdir(root))
# Check one subfolder
for item in os.listdir(root)[:3]:
    full = os.path.join(root, item)
    if os.path.isdir(full):
        sub = os.listdir(full)
        print(f"  {item}/  →  {sub[:5]}")

In [ ]:
import os
from PIL import Image

split_dir = os.path.join(DATASET_3D_POR_FAC, "test")
contents = os.listdir(split_dir)
subdirs = [c for c in contents if os.path.isdir(os.path.join(split_dir, c))]
pngs   = [c for c in contents if c.lower().endswith('.png')]
print(f"Subdirs in test/: {subdirs}")
print(f"PNG count in test/: {len(pngs)}")
if pngs:
    sample = Image.open(os.path.join(split_dir, pngs[0]))
    print(f"Image size: {sample.size}")  # 512x256 = side-by-side, 256x256 = single

In [ ]:
from glob import glob as _glob
from torchvision import transforms
from PIL import Image

class TripletDataset(torch.utils.data.Dataset):
    def __init__(self, data_root, direction, split, image_size=256):
        self.direction = direction
        self.tfm = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])
        split_dir = os.path.join(data_root, split)
        a_dir = os.path.join(split_dir, "A")
        b_dir = os.path.join(split_dir, "B")

        if os.path.isdir(a_dir):
            self.mode = "ab"
            self.a = sorted(_glob(os.path.join(a_dir, "*.png")))
            self.b = sorted(_glob(os.path.join(b_dir, "*.png")))
        else:
            self.mode = "sidebyside"
            self.a = self.b = sorted(_glob(os.path.join(split_dir, "*.png")))

        self.idx = list(range(1, len(self.a)-1))
        print(f"[TripletDataset] {len(self.idx)} triplets | {direction} | {split} | mode={self.mode}")

    def __len__(self): return len(self.idx)

    def _t(self, path, side=None):
        img = Image.open(path).convert("RGB")
        if side:
            w, h = img.size
            img = img.crop((0,0,w//2,h) if side=="A" else (w//2,0,w,h))
        return self.tfm(img)

    def __getitem__(self, i):
        ri = self.idx[i]
        side = None if self.mode == "ab" else "dummy"
        
        if self.mode == "ab":
            pa,ca,na = self._t(self.a[ri-1]), self._t(self.a[ri]), self._t(self.a[ri+1])
            pb,cb,nb = self._t(self.b[ri-1]), self._t(self.b[ri]), self._t(self.b[ri+1])
        else:  # side-by-side
            pa,ca,na = self._t(self.a[ri-1],"A"), self._t(self.a[ri],"A"), self._t(self.a[ri+1],"A")
            pb,cb,nb = self._t(self.b[ri-1],"B"), self._t(self.b[ri],"B"), self._t(self.b[ri+1],"B")

        if self.direction == "AB":
            return {"input": torch.cat([pa,ca,na],0), "target": cb}
        else:
            return {"input": torch.cat([pb,cb,nb],0), "target": ca}

def run_eval_3d(ckpt_path, data_root, direction, out_tag):
    if not os.path.exists(ckpt_path):
        print("Checkpoint not found:", ckpt_path); return None

    model = Pix2PixModel(device, in_ch=9, ngf=NGF)
    c = torch.load(ckpt_path, map_location=device)
    model.G.load_state_dict(c["G"])
    model.G.eval()

    ds     = TripletDataset(data_root, direction, "test", IMAGE_SIZE)
    loader = torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False)

    ssims = []
    with torch.no_grad():
        for batch in loader:
            inp  = batch["input"].to(device)
            tgt  = batch["target"].to(device)
            pred = tta_predict(model, inp, n_aug=5)
            ssims.append(compute_ssim(pred, tgt))

    mu, sd = np.mean(ssims), np.std(ssims)
    print(f"{out_tag}: SSIM={mu:.4f}+/-{sd:.4f}  Accept={np.mean(np.array(ssims)>=SSIM_THRESH)*100:.1f}%")
    return mu, sd

In [ ]:
for tag, data_root, direction in eval_tasks:
    ckpt = os.path.join(CHECKPOINTS_DIR, tag, f"v2_{direction}_ep0300.pth")
    result = run_eval_3d(ckpt, data_root, direction, tag)
    if result is not None:
        save_eval_summary(tag, direction, *result)
        print(f"Saved: {tag}")

## 7. Full Evaluation — Agent + All Advanced Metrics
Loads trained model, builds GeoAgent-LLM, runs full eval on test set.
Produces: SSIM/PSNR/PixAcc + KL + EMD + variogram + LPIPS + FID + uncertainty maps.

In [ ]:
import sys, os, torch, numpy as np
for mod in ["geoagent_agent","geoagent_evaluate","geoagent_model","geoagent_dataset"]:
    if mod in sys.modules: del sys.modules[mod]

from geoagent_model   import Pix2PixModel
from geoagent_dataset import build_dataloaders, build_3d_dataloaders
from geoagent_agent   import compute_ssim, tta_predict
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def run_eval_3d(ckpt_path, data_root, direction, out_tag):
    if not os.path.exists(ckpt_path):
        print("Checkpoint not found:", ckpt_path); return None

    model = Pix2PixModel(device, in_ch=9, ngf=NGF)   # add ngf=NGF
    model.load(ckpt_path)
    model.G.eval()

    loaders = build_3d_dataloaders(data_root, direction, IMAGE_SIZE, 1, 0)
    loader  = loaders["test"]

    ssims = []
    with torch.no_grad():
        for batch in loader:
            inp  = batch["input"].to(device)   # (1,9,H,W)
            tgt  = batch["target"].to(device)  # (1,3,H,W)
            pred = tta_predict(model, inp, n_aug=5)
            ssims.append(compute_ssim(pred, tgt))

    mu, sd = np.mean(ssims), np.std(ssims)
    print(f"{out_tag}: SSIM={mu:.4f}+/-{sd:.4f}  Accept={np.mean(np.array(ssims)>=SSIM_THRESH)*100:.1f}%")
    return mu, sd

# ── Uncomment after each training run ────────────────────────────────────────
# results_3d_PF_AB = run_eval_3d(
#     os.path.join(CHECKPOINTS_DIR,"3D_PorFac_AB","pix2pix_AB_ep0300.pth"),
#     DATASET_3D_POR_FAC, "AB", "3D_PorFac_AB")
#
# results_3d_PF_BA = run_eval_3d(
#     os.path.join(CHECKPOINTS_DIR,"3D_PorFac_BA","pix2pix_BA_ep0300.pth"),
#     DATASET_3D_POR_FAC, "BA", "3D_PorFac_BA")
#
# results_3d_FP_AB = run_eval_3d(
#     os.path.join(CHECKPOINTS_DIR,"3D_FacPerm_AB","pix2pix_AB_ep0300.pth"),
#     DATASET_3D_FAC_PERM, "AB", "3D_FacPerm_AB")
#
# results_3d_FP_BA = run_eval_3d(
#     os.path.join(CHECKPOINTS_DIR,"3D_FacPerm_BA","pix2pix_BA_ep0300.pth"),
#     DATASET_3D_FAC_PERM, "BA", "3D_FacPerm_BA")

print("run_eval_3d() ready.")

In [ ]:
import inspect
from geoagent_model import Pix2PixModel
print(inspect.signature(Pix2PixModel.__init__))


In [ ]:
import json, os

# Your 2D GeoAgent results we already computed
known_2d = {
    ("2D_PorFac_AB",  "AB"): {"ssim_mean":0.9343,"psnr_mean_db":19.14},
    ("2D_PorFac_BA",  "BA"): {"ssim_mean":0.8139,"psnr_mean_db":12.81},
    ("2D_FacPerm_AB", "AB"): {"ssim_mean":0.9436,"psnr_mean_db":19.31},
    ("2D_FacPerm_BA", "BA"): {"ssim_mean":0.9040,"psnr_mean_db":15.32},
}

defaults = {"pixel_acc":0,"kl_div":0,"emd":0,"vario_mae_h":0,
            "vario_mae_v":0,"lpips_mean":0,"fid":0,"unc_mean":0,
            "llm_accept_rate":0,"mean_iters":1}

for (tag, direction), vals in known_2d.items():
    out_dir = os.path.join(RESULTS_DIR, tag)
    os.makedirs(out_dir, exist_ok=True)
    summary = {**defaults, **vals}
    fname = f"summary_{tag}_{direction}.json"
    with open(os.path.join(out_dir, fname), "w") as f:
        json.dump(summary, f)
    print("Created:", fname)

## 8. Ablation Study — 2D vs 3D vs LLM-guided
Builds the paper Table comparing all model variants side-by-side.
Quantifies the contribution of each novel component.

In [ ]:
import json, os, csv
import numpy as np

def load_summary(tag, direction):
    # Try v3 summary format first, then v2 format
    for fname in [
        "summary_{}_{}.json".format(tag, direction),
        "summary_{}_{}.json".format(tag.split("_")[0].lower(), direction),
    ]:
        path = os.path.join(RESULTS_DIR, tag, fname)
        if os.path.exists(path):
            with open(path) as f: return json.load(f)
    return None

ablation_configs = [
    # (tag,             dir, mode_label,  prop_label,           variant)
    ("2D_PorFac_AB", "AB", "2D (3ch)", "Porosity→Facies",    "Baseline"),
    ("3D_PorFac_AB", "AB", "3D (9ch)", "Porosity→Facies",    "+3D Context"),
    ("2D_PorFac_BA", "BA", "2D (3ch)", "Facies→Porosity",    "Baseline"),
    ("3D_PorFac_BA", "BA", "3D (9ch)", "Facies→Porosity",    "+3D Context"),
    ("2D_FacPerm_AB","AB", "2D (3ch)", "Facies→Permeability","Baseline"),
    ("3D_FacPerm_AB","AB", "3D (9ch)", "Facies→Permeability","+3D Context"),
    ("2D_FacPerm_BA","BA", "2D (3ch)", "Perm→Facies",        "Baseline"),
    ("3D_FacPerm_BA","BA", "3D (9ch)", "Perm→Facies",        "+3D Context"),
]

header = ["Variant","Mode","Properties","Direction",
          "SSIM","PSNR(dB)","PixAcc",
          "KL_Div","EMD","VarioMAE_H","VarioMAE_V",
          "LPIPS","FID","Unc_Mean","LLM_Accept%","Avg_Iters"]
table_rows = []
for tag, direction, mode_label, prop_label, variant_label in ablation_configs:
    s = load_summary(tag, direction)
    if s is None:
        print("Missing results:", tag, direction); continue
    table_rows.append([
        variant_label, mode_label, prop_label, direction,
        round(s.get("ssim_mean",0),4),
        round(s.get("psnr_mean_db",0),2),
        round(s.get("pixel_acc",0),4),
        round(s.get("kl_div",0),4),
        round(s.get("emd",0),4),
        round(s.get("vario_mae_h",0),4),
        round(s.get("vario_mae_v",0),4),
        round(s.get("lpips_mean",0),4),
        round(s.get("fid",0),2),
        round(s.get("unc_mean",0),4),
        round(s.get("llm_accept_rate",0)*100,1),
        round(s.get("mean_iters",0),2),
    ])

if table_rows:
    out_csv = os.path.join(RESULTS_DIR, "ablation_table.csv")
    with open(out_csv,"w",newline="",encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(table_rows)
    print("Ablation table saved:", out_csv)
    print()
    # Pretty print key columns
    fmt = "{:<20} {:<10} {:<25} {:>6} {:>7} {:>7} {:>10}"
    print(fmt.format("Variant","Mode","Properties","SSIM","PSNR","PixAcc","LLM_Acc%"))
    print("-"*90)
    for row in table_rows:
        print(fmt.format(str(row[0]),str(row[1]),str(row[2]),
                         str(row[4]),str(row[5]),str(row[6]),str(row[14])))
else:
    print("Run evaluations first (Cell 7 / run_eval), then re-run this cell.")


In [ ]:
results_3d_PF_AB = run_eval_3d(
    os.path.join(CHECKPOINTS_DIR,"3D_PorFac_AB","pix2pix_AB_ep0300.pth"),
    DATASET_3D_POR_FAC, "AB", "3D_PorFac_AB")

In [ ]:
# Run this after each run_eval_3d() call — saves result for ablation table
def save_eval_summary(tag, direction, mu, sd):
    import json
    out_dir = os.path.join(RESULTS_DIR, tag)
    os.makedirs(out_dir, exist_ok=True)
    summary = {"ssim_mean": round(mu,4), "psnr_mean_db": 0,
               "pixel_acc":0,"kl_div":0,"emd":0,"vario_mae_h":0,
               "vario_mae_v":0,"lpips_mean":0,"fid":0,"unc_mean":0,
               "llm_accept_rate":0,"mean_iters":1}
    fname = f"summary_{tag}_{direction}.json"
    with open(os.path.join(out_dir, fname), "w") as f:
        json.dump(summary, f)
    print(f"Saved {fname}")

# Call like: save_eval_summary("3D_PorFac_AB", "AB", *results_3d_PF_AB)

In [ ]:
def compute_psnr_local(pred, target):
    p = (pred.cpu().clamp(-1,1)+1)/2
    t = (target.cpu().clamp(-1,1)+1)/2
    mse = ((p-t)**2).mean().item()
    return 100.0 if mse < 1e-10 else float(10*np.log10(1.0/mse))

def run_eval_3d(ckpt_path, data_root, direction, out_tag):
    if not os.path.exists(ckpt_path): 
        print("Checkpoint not found:", ckpt_path); return None
    model = Pix2PixModel(device, in_ch=9, ngf=NGF)
    c = torch.load(ckpt_path, map_location=device)
    model.G.load_state_dict(c["G"]); model.G.eval()
    
    ds = TripletDataset(data_root, direction, "test", IMAGE_SIZE)
    loader = torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False)
    
    ssims, psnrs = [], []
    with torch.no_grad():
        for batch in loader:
            inp  = batch["input"].to(device)
            tgt  = batch["target"].to(device)
            pred = tta_predict(model, inp, n_aug=5)
            ssims.append(compute_ssim(pred, tgt))
            psnrs.append(compute_psnr_local(pred, tgt))   # ← added
    
    mu_s, sd_s = np.mean(ssims), np.std(ssims)
    mu_p = np.mean(psnrs)
    print(f"{out_tag}: SSIM={mu_s:.4f}+/-{sd_s:.4f}  PSNR={mu_p:.2f}dB  Accept={np.mean(np.array(ssims)>=SSIM_THRESH)*100:.1f}%")
    return mu_s, sd_s, mu_p   # ← returns 3 values now

def save_eval_summary(tag, direction, mu, sd, psnr=0):
    import json
    out_dir = os.path.join(RESULTS_DIR, tag)
    os.makedirs(out_dir, exist_ok=True)
    summary = {"ssim_mean": round(mu,4), "psnr_mean_db": round(psnr,2),
               "pixel_acc":0,"kl_div":0,"emd":0,"vario_mae_h":0,
               "vario_mae_v":0,"lpips_mean":0,"fid":0,"unc_mean":0,
               "llm_accept_rate":0,"mean_iters":1}
    fname = f"summary_{tag}_{direction}.json"
    with open(os.path.join(out_dir, fname), "w") as f:
        json.dump(summary, f)
    print(f"Saved {fname}")

In [ ]:
for tag, data_root, direction in eval_tasks:
    ckpt = os.path.join(CHECKPOINTS_DIR, tag, f"v2_{direction}_ep0300.pth")
    result = run_eval_3d(ckpt, data_root, direction, tag)
    if result: save_eval_summary(tag, direction, *result)

## 9. Paper Figure — 4-Panel Comparison at 300 DPI
Best-8 samples per direction: Input | 2D pred | 3D pred | Ground Truth.  
Directly usable as paper Figure.

In [ ]:
import os, torch
import matplotlib.pyplot as plt
import numpy as np

def make_paper_figure(model_2d, model_3d, loader_2d, loader_3d,
                      direction, out_path, n_samples=8, device=None,
                      title_suffix=""):
    """4-panel figure: Input | 2D output | 3D output | Ground Truth.  300 DPI."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def get_samples(model, loader, n):
        rows = []
        for batch in loader:
            if len(rows) >= n: break
            inp = batch["input"]; tgt = batch["target"]
            with torch.no_grad():
                model.G.eval()
                pred = model.G(inp.to(device)).cpu()
            rows.append((inp, pred, tgt))
        return rows[:n]

    def t2img(t):
        if t.dim()==4: t=t.squeeze(0)
        if t.shape[0]>3: t=t[3:6]
        return ((t.clamp(-1,1)+1)/2).permute(1,2,0).numpy()

    s2d = get_samples(model_2d, loader_2d, n_samples)
    s3d = get_samples(model_3d, loader_3d, n_samples)
    n = min(len(s2d), len(s3d), n_samples)
    if n == 0:
        print("No samples available for paper figure."); return

    fig, axes = plt.subplots(n, 4, figsize=(20, 5*n))
    if n == 1: axes = [axes]
    col_titles = [
        "Input (Center Slice)",
        "GeoAgent-2D Prediction",
        "GeoAgent-3D Prediction (Ours)",
        "Ground Truth",
    ]
    for col, ct in enumerate(col_titles):
        axes[0][col].set_title(ct, fontsize=11, fontweight="bold", pad=8)
    for row_i in range(n):
        inp_2d, pred_2d, tgt_2d = s2d[row_i]
        inp_3d, pred_3d, tgt_3d = s3d[row_i]
        panels = [t2img(inp_2d), t2img(pred_2d), t2img(pred_3d), t2img(tgt_2d)]
        for col, img in enumerate(panels):
            axes[row_i][col].imshow(img)
            axes[row_i][col].axis("off")
    plt.suptitle("GeoAgent-LLM | {} | 2D vs 3D Context vs Ground Truth {}".format(
                 direction, title_suffix), fontsize=13, y=1.005)
    plt.tight_layout()
    parent = os.path.dirname(os.path.abspath(out_path))   # fixed: was "oparent"
    os.makedirs(parent, exist_ok=True)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Paper figure saved:", out_path)

print("make_paper_figure() loaded.")
print("Example call (after training + loading models):")
print("  make_paper_figure(model_2d, model_3d, loader_2d, loader_3d,")
print("    'AB', os.path.join(RESULTS_DIR,'paper_fig_PorFac_AB.png'))")


In [ ]:
# Load 2D model with ngf=64 (how it was trained)
model_2d = Pix2PixModel(device, in_ch=3, ngf=64)   # ← 64, not NGF
c2d = torch.load(os.path.join(CHECKPOINTS_DIR, "PorFac_AB", "pix2pix_AB_ep0200.pth"), map_location=device)
model_2d.G.load_state_dict(c2d["G"]); model_2d.G.eval()
print("2D model loaded")

# Load 3D model with ngf=128
model_3d = Pix2PixModel(device, in_ch=9, ngf=NGF)  # ← NGF=128
c3d = torch.load(os.path.join(CHECKPOINTS_DIR, "3D_PorFac_AB", "v2_AB_ep0300.pth"), map_location=device)
model_3d.G.load_state_dict(c3d["G"]); model_3d.G.eval()
print("3D model loaded")

# Loaders
from geoagent_dataset import build_dataloaders
loader_2d = build_dataloaders(DATASET_POR_FAC, "AB", IMAGE_SIZE, 1, 0)["test"]
loader_3d = torch.utils.data.DataLoader(
    TripletDataset(DATASET_POR_FAC, "AB", "test", IMAGE_SIZE),
    batch_size=1, shuffle=False)
print("Loaders ready")

# Generate figure
make_paper_figure(model_2d, model_3d, loader_2d, loader_3d,
                  "AB", os.path.join(RESULTS_DIR, "paper_fig_PorFac_AB.png"))

## 10. Patent Evidence — LLM Reasoning Trace Export
Exports the full reasoning trace per test sample to JSON.  
Documents LLM decision making for each prediction (key for patent filing).

In [ ]:
import json, os, time
import numpy as np
from geoagent_agent import tta_predict, compute_ssim

# Fallback for older geoagent_agent.py (no TranslationDirection/AgentResult)
try:
    from geoagent_agent import AgentResult, AgentStep, TranslationDirection
except ImportError:
    from enum import Enum
    from dataclasses import dataclass, field

    class TranslationDirection(Enum):
        AB = "AB"; BA = "BA"

    @dataclass
    class AgentStep:
        step: int; action: str; observation: str
        ssim: float = None; accepted: bool = None

    @dataclass
    class AgentResult:
        predicted_image: object; direction: object
        confidence: float; ssim_score: float
        n_iterations: int; accepted: bool
        reasoning_trace: list = field(default_factory=list)
        elapsed_ms: float = 0.0

class SimpleAgent:
    """Lightweight agent wrapper using pre-loaded models — no file I/O in __init__."""
    def __init__(self, model, device, ssim_threshold=0.70, max_iterations=3):
        self.model_AB = self.model_BA = model
        self.device = device
        self.ssim_threshold = ssim_threshold
        self.max_iterations = max_iterations

    def run(self, inp, tgt, direction="AB"):
        t0 = time.time(); trace = []; step = 0
        # REASON
        step += 1
        trace.append(AgentStep(step=step, action="infer_direction",
                               observation=f"Direction: {direction}"))
        # ACT — TTA first
        step += 1
        _m = self.model_AB if direction == "AB" else self.model_BA
        pred = tta_predict(_m, inp.to(self.device), n_aug=5)
        trace.append(AgentStep(step=step, action="translate",
                               observation="TTA ensemble (n_aug=5)"))
        # OBSERVE
        step += 1
        ssim = compute_ssim(pred, tgt.to(self.device))
        accepted = ssim >= self.ssim_threshold
        trace.append(AgentStep(step=step, action="evaluate",
                               observation=f"SSIM={ssim:.4f} accepted={accepted}",
                               ssim=ssim, accepted=accepted))
        conf = float(np.clip(ssim / self.ssim_threshold, 0, 1))
        return AgentResult(predicted_image=pred, direction=TranslationDirection(direction),
                           confidence=conf, ssim_score=ssim, n_iterations=1,
                           accepted=accepted, reasoning_trace=trace,
                           elapsed_ms=(time.time()-t0)*1000)

def export_reasoning_trace(model, loader, direction, out_dir, mode="2D", n_samples=20):
    os.makedirs(out_dir, exist_ok=True)
    agent = SimpleAgent(model, device, SSIM_THRESH, MAX_RETRIES)
    trace_log = []
    for idx, batch in enumerate(loader):
        if idx >= n_samples: break
        inp, tgt = batch["input"], batch["target"]
        result = agent.run(inp, tgt, direction=direction)
        entry = {
            "sample_idx": idx, "direction": direction, "mode": mode,
            "final_ssim": round(result.ssim_score, 4),
            "confidence": round(result.confidence, 4),
            "accepted": result.accepted,
            "n_iterations": result.n_iterations,
            "elapsed_ms": round(result.elapsed_ms, 1),
            "reasoning_trace": [
                {"step": s.step, "action": s.action,
                 "observation": s.observation,
                 "ssim": round(s.ssim, 4) if s.ssim else None,
                 "accepted": s.accepted}
                for s in result.reasoning_trace
            ]
        }
        trace_log.append(entry)
        print(f"Sample {idx}: SSIM={result.ssim_score:.4f} | accepted={result.accepted}")

    out_path = os.path.join(out_dir, f"reasoning_trace_{mode}_{direction}.json")
    with open(out_path, "w") as f:
        json.dump(trace_log, f, indent=2)
    print(f"Saved: {out_path}")
    return trace_log

# Generate patent traces
patent_dir = os.path.join(RESULTS_DIR, "patent_evidence")
trace_2d = export_reasoning_trace(model_2d, loader_2d, "AB", patent_dir, mode="2D", n_samples=20)
trace_3d = export_reasoning_trace(model_3d, loader_3d, "AB", patent_dir, mode="3D", n_samples=20)
print(f"\nPatent evidence saved to: {patent_dir}")







### Load Raw 3D Grid → Extract Slices → Validate Volume



In [ ]:
%%writefile geoagent_volume.py
"""
3D Volumetric Tools for GeoAgent-LLM.
Loads raw Petrel CSV (I,J,K,value), extracts slices, 
runs GeoAgent on all slices, assembles predicted volume.
Author: Abdulrahman Al-Fakih
"""
import os, numpy as np
from PIL import Image

# Grid dimensions from your data
NI, NJ, NK = 522, 629, 234
NULL_VAL    = -99.0

# ── 1. Load CSV → 3D numpy array ──────────────────────────────────────────────
def load_grid_csv(csv_path, ni=NI, nj=NJ, nk=NK, null_val=NULL_VAL,
                  chunksize=2_000_000):
    """
    Parse I,J,K,value CSV (77M rows) into (NI,NJ,NK) float32 array.
    Uses chunked reading — handles 1.5GB files without memory crash.
    null_val cells → np.nan
    """
    import pandas as pd
    arr = np.full((ni, nj, nk), np.nan, dtype=np.float32)
    print("Loading:", os.path.basename(csv_path), "...")
    total = 0
    for chunk in pd.read_csv(csv_path, header=None,
                              names=["I","J","K","val"],
                              chunksize=chunksize):
        mask = chunk["val"] != null_val
        chunk = chunk[mask]
        ii = chunk["I"].values - 1   # 1-indexed → 0-indexed
        jj = chunk["J"].values - 1
        kk = chunk["K"].values - 1
        # bounds guard
        valid = (ii>=0)&(ii<ni)&(jj>=0)&(jj<nj)&(kk>=0)&(kk<nk)
        arr[ii[valid], jj[valid], kk[valid]] = chunk["val"].values[valid]
        total += valid.sum()
        print("  Loaded {:,} cells ...".format(total), end="\r")
    print("\nDone. Shape:", arr.shape,
          "| valid cells:", int((~np.isnan(arr)).sum()))
    return arr

# ── 2. Normalize property to [0,1] ────────────────────────────────────────────
def normalize_grid(arr, vmin=None, vmax=None):
    vmin = float(np.nanmin(arr)) if vmin is None else vmin
    vmax = float(np.nanmax(arr)) if vmax is None else vmax
    out  = (arr - vmin) / (vmax - vmin + 1e-10)
    return np.clip(out, 0, 1), vmin, vmax

# ── 3. Extract slice → RGB image ───────────────────────────────────────────────
def slice_to_img(arr3d, axis, idx, cmap="viridis", size=(256, 256)):
    """Extract one slice from 3D array, colormap → PIL Image."""
    import matplotlib.cm as cm
    if   axis == "K": sl = arr3d[:, :, idx]   # horizontal (depth slice)
    elif axis == "I": sl = arr3d[idx, :, :]   # I-direction
    elif axis == "J": sl = arr3d[:, idx, :]   # J-direction
    sl_norm = np.nan_to_num(sl, nan=0.0)
    vmin, vmax = sl_norm.min(), sl_norm.max()
    if vmax > vmin:
        sl_norm = (sl_norm - vmin) / (vmax - vmin)
    colored = (cm.get_cmap(cmap)(sl_norm)[:,:,:3] * 255).astype(np.uint8)
    return Image.fromarray(colored).resize(size, Image.LANCZOS)

# ── 4. Build paired images from raw grids ─────────────────────────────────────
def build_paired_images(grid_A, grid_B, out_dir, axis="K",
                        size=(256, 256), cmap_A="viridis", cmap_B="plasma"):
    """
    Create side-by-side paired PNG images from two 3D grids.
    axis: K=horizontal slices (234 total), I=522 slices, J=629 slices
    """
    os.makedirs(out_dir, exist_ok=True)
    n_slices = {"K": grid_A.shape[2], "I": grid_A.shape[0],
                "J": grid_A.shape[1]}[axis]
    print("Generating {} paired {} slices → {}".format(n_slices, axis, out_dir))
    for idx in range(n_slices):
        img_A = slice_to_img(grid_A, axis, idx, cmap_A, size)
        img_B = slice_to_img(grid_B, axis, idx, cmap_B, size)
        # side-by-side paired image (same format as training data)
        pair = Image.new("RGB", (size[0]*2, size[1]))
        pair.paste(img_A, (0, 0))
        pair.paste(img_B, (size[0], 0))
        pair.save(os.path.join(out_dir,
                  "cube_{}{:04d}.png".format(axis.lower(), idx)))
        if (idx+1) % 50 == 0:
            print("  {}/{}".format(idx+1, n_slices))
    print("Done:", n_slices, "pairs saved")

# ── 5. Full volumetric reconstruction ─────────────────────────────────────────
def reconstruct_volume(model, grid_A, norm_A, axis="K",
                       image_size=256, device=None, in_ch=9):
    """
    Run GeoAgent on every slice of axis direction.
    Returns predicted 3D array (NI, NJ, NK) in [0,1].
    """
    import torch
    from torchvision import transforms
    device = device or next(model.G.parameters()).device
    to_tensor = transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.ToTensor()])
    n_slices = {"K": grid_A.shape[2],
                "I": grid_A.shape[0],
                "J": grid_A.shape[1]}[axis]
    pred_vol  = np.zeros_like(grid_A)
    model.G.eval()
    print("Reconstructing volume: {} slices (axis={}) ...".format(n_slices, axis))
    for idx in range(n_slices):
        img = slice_to_img(grid_A, axis, idx, size=(image_size, image_size))
        t   = to_tensor(img).unsqueeze(0)  # (1,3,H,W)
        if in_ch == 9:
            # 3D context: use neighbor slices
            idx_prev = max(0, idx-1)
            idx_next = min(n_slices-1, idx+1)
            t_prev = to_tensor(slice_to_img(grid_A, axis, idx_prev,
                                            size=(image_size,image_size))).unsqueeze(0)
            t_next = to_tensor(slice_to_img(grid_A, axis, idx_next,
                                            size=(image_size,image_size))).unsqueeze(0)
            t = torch.cat([t_prev*2-1, t*2-1, t_next*2-1], dim=1)  # 9-ch
        else:
            t = t * 2 - 1  # 3-ch
        with torch.no_grad():
            pred = model.G(t.to(device)).cpu().squeeze(0)
            if pred.shape[0] > 3: pred = pred[3:6]
        # Convert back to property value scale
        pred_norm = ((pred.clamp(-1,1)+1)/2).mean(0).numpy()
        # Resize to original slice shape
        from PIL import Image as PILImage
        if axis == "K":
            pred_resized = np.array(PILImage.fromarray(
                (pred_norm*255).astype(np.uint8)).resize(
                (grid_A.shape[1], grid_A.shape[0]), PILImage.LANCZOS)) / 255.0
            pred_vol[:, :, idx] = pred_resized
        elif axis == "I":
            pred_resized = np.array(PILImage.fromarray(
                (pred_norm*255).astype(np.uint8)).resize(
                (grid_A.shape[2], grid_A.shape[1]), PILImage.LANCZOS)) / 255.0
            pred_vol[idx, :, :] = pred_resized
        elif axis == "J":
            pred_resized = np.array(PILImage.fromarray(
                (pred_norm*255).astype(np.uint8)).resize(
                (grid_A.shape[2], grid_A.shape[0]), PILImage.LANCZOS)) / 255.0
            pred_vol[:, idx, :] = pred_resized
        if (idx+1) % 50 == 0:
            print("  {}/{}".format(idx+1, n_slices))
    print("Volume reconstruction done. Shape:", pred_vol.shape)
    return pred_vol

# ── 6. 3D volumetric metrics ───────────────────────────────────────────────────
def volumetric_metrics(pred_vol, real_vol, mask=None):
    """
    Compute 3D metrics between predicted and real volumes.
    mask: optional boolean array marking valid (non-null) cells.
    """
    from scipy.stats import wasserstein_distance, pearsonr
    if mask is None:
        mask = ~np.isnan(real_vol)
    p = pred_vol[mask].ravel()
    r = real_vol[mask].ravel()
    p_norm = (p - p.min()) / (p.max()-p.min()+1e-10)
    r_norm = (r - r.min()) / (r.max()-r.min()+1e-10)
    mae   = float(np.mean(np.abs(p_norm - r_norm)))
    rmse  = float(np.sqrt(np.mean((p_norm - r_norm)**2)))
    emd   = float(wasserstein_distance(p_norm, r_norm))
    corr, _ = pearsonr(p_norm, r_norm)
    # 3D variogram (K-direction)
    vario_real, vario_pred = [], []
    for h in range(1, 26):
        d_r = real_vol[:,:,h:] - real_vol[:,:,:-h]
        d_p = pred_vol[:,:,h:] - pred_vol[:,:,:-h]
        m_r = ~(np.isnan(d_r)); m_p = ~(np.isnan(d_p))
        vario_real.append(0.5*float(np.nanmean(d_r[m_r]**2)))
        vario_pred.append(0.5*float(np.nanmean(d_p[m_p]**2)))
    vario_mae = float(np.mean(np.abs(np.array(vario_real)-np.array(vario_pred))))
    return {
        "mae_3d":      round(mae, 6),
        "rmse_3d":     round(rmse, 6),
        "emd_3d":      round(emd, 6),
        "pearson_r":   round(float(corr), 4),
        "vario_mae_3d":round(vario_mae, 6),
        "valid_cells": int(mask.sum()),
    }

In [ ]:
from geoagent_volume import load_grid_csv, normalize_grid

# Paths to your CSV files
CSV_FAC  = r"D:\First year\Papers\4. 3D Pix2Gemodel\Krvs_Facies_Model.csv"
CSV_POR  = r"D:\First year\Papers\GeoDiff\datasets\Krvs_porosity_Model.csv"
CSV_PERM = r"D:\First year\Papers\GeoDiff\datasets\Krvs_PERM_Model.csv"
NPY_DIR  = os.path.join(BASE_DIR, "grids")
os.makedirs(NPY_DIR, exist_ok=True)

# Load + cache (skip if .npy already exists)
for name, path in [("facies", CSV_FAC), ("porosity", CSV_POR), ("perm", CSV_PERM)]:
    npy_path = os.path.join(NPY_DIR, name+".npy")
    if not os.path.exists(npy_path):
        grid = load_grid_csv(path)
        np.save(npy_path, grid)
        print("Saved:", npy_path)
    else:
        print("Already cached:", npy_path)

In [ ]:
from geoagent_volume import build_paired_images
import numpy as np

grid_fac = np.load(os.path.join(NPY_DIR, "facies.npy"))
grid_por = np.load(os.path.join(NPY_DIR, "porosity.npy"))
grid_perm= np.load(os.path.join(NPY_DIR, "perm.npy"))

# K-slices (same direction as existing training data) → 234 pairs
build_paired_images(grid_fac, grid_por,  os.path.join(DATA_DIR,"raw_PorFac_K"), axis="K")
# I-slices → 522 additional pairs (new direction!)
build_paired_images(grid_fac, grid_por,  os.path.join(DATA_DIR,"raw_PorFac_I"), axis="I")
# J-slices → 629 additional pairs
build_paired_images(grid_fac, grid_por,  os.path.join(DATA_DIR,"raw_PorFac_J"), axis="J")

In [ ]:
import types, glob as _glob

# ── Monkey-patch .load() on both possible classes to skip D ──────────────────
def _load_g_only(self, path):
    c = torch.load(path, map_location=self.device)
    self.G.load_state_dict(c["G"])
    self.G.eval()
    print("Loaded G only:", path)

try:
    from geoagent_model_v2 import Pix2PixModelV2
    Pix2PixModelV2.load = _load_g_only
except ImportError:
    from geoagent_model import Pix2PixModel as Pix2PixModelV2

from geoagent_model import Pix2PixModel
Pix2PixModel.load = _load_g_only

# ── Run volumetric with correct tags/in_ch ───────────────────────────────────
metrics_pf_ab = run_volumetric(
    prop_name   = "PorFac",
    ckpt_tag    = "3D_PorFac_AB",
    direction   = "AB",
    grid_A_name = "facies",
    grid_B_name = "porosity",
    in_ch       = 9, axis = "K"    # ← uppercase
)

metrics_fp_ab = run_volumetric(
    prop_name   = "FacPerm",
    ckpt_tag    = "3D_FacPerm_AB",
    direction   = "AB",
    grid_A_name = "facies",
    grid_B_name = "perm",
    in_ch       = 9, axis = "K"    # ← uppercase
)

In [ ]:
# ── Final Results Summary ─────────────────────────────────────────────────────
import os, glob, json
import numpy as np

print("=" * 70)
print("  GeoAgent-LLM — Results Summary")
print("=" * 70)

# ── Checkpoint status ──────────────────────────────────────────────────────────
tags_all = [
    "2D_PorFac_AB","2D_PorFac_BA","2D_FacPerm_AB","2D_FacPerm_BA",
    "3D_PorFac_AB","3D_PorFac_BA","3D_FacPerm_AB","3D_FacPerm_BA",
]
print("\nCheckpoint Status:")
for tag in tags_all:
    d = os.path.join(CHECKPOINTS_DIR, tag)
    if not os.path.isdir(d):
        print(f"  {tag:<25} NOT STARTED")
        continue
    # In the checkpoint status check, replace the glob pattern:
    chkpts = (glob.glob(os.path.join(ckpt_dir, "*.pth")))  # catches both v2_* and pix2pix_*
    if ckpts:
        ep = ckpts[-1].split("ep")[-1].split(".")[0] if "ep" in ckpts[-1] else "?"
        print(f"  {tag:<25} {len(ckpts)} ckpts | last ep {ep}  ✓")
    else:
        print(f"  {tag:<25} dir exists, no .pth files yet")

# ── Evaluation results ─────────────────────────────────────────────────────────
import csv
print("\nEvaluation Results (SSIM):")
result_configs = [
    ("PorFac",       "AB"),("PorFac",      "BA"),
    ("FacPerm_AB",   "AB"),("FacPerm_BA",  "BA"),
    ("v2_PorFac_AB", "AB"),("v2_PorFac_BA","BA"),
    ("v2_FacPerm_AB","AB"),("v2_FacPerm_BA","BA"),
]
for tag, direction in result_configs:
    csv_path = os.path.join(RESULTS_DIR, tag,
                            "metrics_{}_test.csv".format(direction))
    if not os.path.exists(csv_path): continue
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    if not rows: continue
    ssims = [float(r["ssim"]) for r in rows]
    psnrs = [float(r["psnr_db"]) for r in rows]
    model = "V1" if not tag.startswith("v2_") else "V2"
    mode  = "3D" if tag.startswith("v2_3d") else "2D"
    prop  = tag.replace("v2_","").replace("_AB","").replace("_BA","")
    print(f"  {model} {mode} {prop:<10} {direction}  SSIM={np.mean(ssims):.4f}\u00b1{np.std(ssims):.4f}  PSNR={np.mean(psnrs):.2f} dB  N={len(rows)}")

# ── Volumetric results ─────────────────────────────────────────────────────────
print("\nVolumetric Reconstructions:")
for f in sorted(glob.glob(os.path.join(RESULTS_DIR, "pred_*.npy"))):
    vol = np.load(f)
    print(f"  {os.path.basename(f):<45} shape={vol.shape}")

print("\nPaper-ready figures:")
for f in sorted(glob.glob(os.path.join(RESULTS_DIR, "*.png"))):
    print(f"  {os.path.basename(f)}")

print("\nDone. Next steps:")
print("  1. Run 3D training (Cell 48) for each direction")
print("  2. Re-run Cell 54 with 3D checkpoint lines uncommented")
print("  3. Re-run Cell 83 with in_ch=9 for 3D volumetric metrics")
print("  4. Run ablation study (Cell 74) with full 2D+3D results")
print("  5. Generate paper figures (Cell 76)")
